# Designing the warehouse and its fact tables

The first notebook (`data_flow_demo.ipynb`) showed how data *moves*: sources → landing → raw →
trusted → refined. This one zooms into the trusted zone and asks the design questions behind it:

1. **Why build a warehouse at all?** What goes wrong if we just query the POS and the Grab export directly?
2. **How did we build ours?** The architecture, the four design steps, the conformance matrix, dimensions with special-case rows, and how a fact row gets loaded.
3. **How do you design a fact table?** Grain and drilling across, snapshots, child facts, additivity, no NULL keys, the four fact types, and why every fact table in our README looks the way it does.

Everything runs in an in-memory DuckDB, so nothing is written to disk. The data is tiny on purpose.
Every number in this notebook can be checked by hand, and that is how you learn the rules.
Each section ends with a **Try it** box: small changes to make so you can see a rule break.

โน้ตบุ๊กแรก (`data_flow_demo.ipynb`) เล่าว่าข้อมูล *ไหล* ยังไง: sources → landing → raw → trusted → refined
โน้ตบุ๊กนี้ซูมเข้าไปที่ trusted zone แล้วตอบคำถามเชิงออกแบบที่อยู่เบื้องหลัง:

1. **ทำ warehouse ไปเพื่ออะไร?** ถ้า query จาก POS กับไฟล์ export ของ Grab ตรง ๆ จะพังตรงไหน
2. **เราทำออกมายังไง?** สถาปัตยกรรม, 4 ขั้นตอนการออกแบบ, conformance matrix, dimension ที่มี special-case row และการโหลด fact หนึ่งแถว
3. **ออกแบบ fact table ยังไง?** grain, additivity, การ snapshot ค่า, child fact, fact 4 ประเภท และเหตุผลว่าทำไม fact ทุกตัวใน README ถึงหน้าตาแบบนั้น

ทุกอย่างรันบน DuckDB ในหน่วยความจำ ไม่เขียนไฟล์ลงดิสก์ ข้อมูลเล็กโดยตั้งใจ
ทุกตัวเลขในโน้ตบุ๊กนี้คิดมือตามได้ และนั่นคือวิธีที่จะเข้าใจกฎแต่ละข้อ
ท้ายแต่ละ section มีกล่อง **Try it** ให้ลองแก้นิดหน่อยเพื่อดูว่ากฎข้อนั้นพังยังไง

```
PART 1  Why a warehouse      ทำไปเพื่ออะไร      sources are hard to ask questions of
PART 2  How we built it      ทำออกมายังไง       architecture → 4 steps → conformance matrix → dimensions → load facts
PART 3  Fact table design    ออกแบบ Fact Table   grain · snapshot · additivity · no NULL keys · types · catalogue
```

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

con = duckdb.connect()                     # in-memory: gone when the kernel stops  |  อยู่ในหน่วยความจำ ปิด kernel แล้วหายไป
for schema in ["src", "stg", "dw"]:        # src = operational systems, stg = cleaned common shape, dw = the warehouse
    con.execute(f"CREATE SCHEMA {schema}")  # src = ระบบหน้าร้าน, stg = ข้อมูลที่ทำความสะอาดแล้ว, dw = warehouse

def q(sql):
    # run SQL and return a DataFrame so Jupyter shows it as a table  |  รัน SQL แล้วคืนเป็น DataFrame ให้ Jupyter แสดงเป็นตาราง
    return con.execute(sql).df()

def load(table, df):
    # copy a pandas DataFrame into a DuckDB table  |  คัดลอก DataFrame ของ pandas ลงเป็นตารางใน DuckDB
    con.register("_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM _tmp")
    con.unregister("_tmp")

print("ready", duckdb.__version__)

ready 2.0.0


---
# PART 1 · Why a warehouse? (ทำไปเพื่ออะไร)

## 1.1 · What the shop's systems actually look like

Two of our five sources are enough to make the point:

- **POS** is an *operational* (OLTP) database. It is normalized, it is built for taking one order fast, and
  it keeps **current state only**. When the owner raised Pepperoni L from 329 to 349 THB on Saturday,
  the POS simply overwrote the price. Modifiers like extra cheese are stored as text on the line.
- **Grab** gives us a flat CSV export: one row per *item*, with order-level fields (delivery fee,
  commission) **repeated on every item row**, timestamps in **UTC**, and its own item names.

Two business days: **Friday 4 Sep 2026** and **Saturday 5 Sep 2026**. The shop trades 10:00–02:00, so an
order at 01:20 on the 5th belongs to *Friday's* business day.

ระบบหน้าร้านจริง ๆ หน้าตาเป็นยังไง: ใช้แค่ 2 จาก 5 แหล่งก็พอจะเห็นภาพ
- **POS** เป็นฐานข้อมูลแบบ *operational* (OLTP) ถูก normalize ออกแบบมาเพื่อรับออร์เดอร์ทีละใบให้เร็ว และเก็บแค่ **สถานะปัจจุบัน**
  วันเสาร์เจ้าของร้านขึ้นราคา Pepperoni L จาก 329 เป็น 349 บาท POS ก็เขียนทับราคาเดิมไปเลย ส่วน modifier อย่างชีสเพิ่ม เก็บเป็น text ในบรรทัดออร์เดอร์
- **Grab** ให้ไฟล์ CSV แบบแบน: หนึ่งแถวต่อหนึ่ง *รายการอาหาร* ฟิลด์ระดับออร์เดอร์ (ค่าส่ง ค่าคอมมิชชัน) **ซ้ำอยู่ทุกแถว**
  เวลาเป็น **UTC** และใช้ชื่อเมนูของ Grab เอง

ข้อมูลมีสองวันทำการ: **ศุกร์ 4 ก.ย. 2026** และ **เสาร์ 5 ก.ย. 2026** ร้านเปิด 10:00–02:00 ออร์เดอร์ตอนตี 1:20 ของวันที่ 5 จึงเป็นของ *วันศุกร์*

In [2]:
# ---- POS: three normalized tables, current state only  |  POS: สามตาราง normalize แล้ว เก็บแค่สถานะปัจจุบัน ----
pos_menu = pd.DataFrame([
    ("PEP-M",  "Pepperoni M",  249),
    ("PEP-L",  "Pepperoni L",  349),     # was 329 until Saturday, overwritten  |  เคยเป็น 329 จนถึงวันเสาร์ ถูกเขียนทับไปแล้ว
    ("HAW-L",  "Hawaiian L",   319),
    ("GARLIC", "Garlic Bread",  89),
    ("COKE",   "Coke",          35),
], columns=["item_id", "name", "price"])

pos_orders = pd.DataFrame([
    ("P-1001", "QR",    "2026-09-04 12:10", "m-081"),
    ("P-1002", "PHONE", "2026-09-04 19:30", "m-092"),
    ("P-1003", "QR",    "2026-09-05 01:20", None),     # after midnight: Friday's business day  |  หลังเที่ยงคืน: นับเป็นวันศุกร์
    ("P-1004", "QR",    "2026-09-05 12:45", None),
    ("P-1005", "PHONE", "2026-09-05 20:05", "m-081"),
], columns=["pos_order_id", "channel", "ordered_at", "member_id"])   # POS clock is Bangkok time  |  นาฬิกา POS เป็นเวลากรุงเทพ

pos_order_items = pd.DataFrame([
    ("P-1001", 1, "PEP-L",  1, 329, "EXTRA_CHEESE+40"),
    ("P-1001", 2, "COKE",   2,  35, ""),
    ("P-1002", 1, "HAW-L",  1, 319, ""),
    ("P-1002", 2, "GARLIC", 1,  89, ""),
    ("P-1003", 1, "PEP-L",  1, 329, "EXTRA_CHEESE+40;NO_CHILI+0"),   # a list hiding in one text column  |  list ซ่อนอยู่ในคอลัมน์ text เดียว
    ("P-1004", 1, "PEP-L",  2, 349, ""),
    ("P-1004", 2, "COKE",   2,  35, ""),
    ("P-1005", 1, "PEP-M",  1, 249, "EXTRA_CHEESE+40"),
    ("P-1005", 2, "HAW-L",  1, 319, ""),
], columns=["pos_order_id", "line_no", "item_id", "qty", "unit_price", "modifiers"])

# ---- Grab: one flat export, order fields repeated per item row, UTC  |  Grab: ไฟล์แบน ฟิลด์ออร์เดอร์ซ้ำทุกแถว เวลาเป็น UTC ----
grab_export = pd.DataFrame([
    ("GF-201", "2026-09-04T11:30:00Z", "Pepperoni Pizza (L)", 1, 379, 25, 146.4),
    ("GF-201", "2026-09-04T11:30:00Z", "Garlic Bread",        1, 109, 25, 146.4),   # same fee and commission again  |  ค่าส่งกับคอมมิชชันซ้ำอีกรอบ
    ("GF-202", "2026-09-04T18:10:00Z", "Hawaiian Pizza (L)",  1, 369, 25, 110.7),   # 18:10 UTC = 01:10 Bangkok, still Friday
    ("GF-203", "2026-09-05T06:00:00Z", "Pepperoni Pizza (L)", 2, 399, 25, 252.9),
    ("GF-203", "2026-09-05T06:00:00Z", "Coke",                1,  45, 25, 252.9),
], columns=["orderID", "orderTime", "itemName", "quantity", "itemPrice", "deliveryFee", "merchantCommission"])

for name, df in [("src.pos_menu", pos_menu), ("src.pos_orders", pos_orders),
                 ("src.pos_order_items", pos_order_items), ("src.grab_export", grab_export)]:
    load(name, df)

display(q("SELECT * FROM src.pos_order_items"))
display(q("SELECT * FROM src.grab_export"))

,pos_order_id,line_no,item_id,qty,unit_price,modifiers
0,P-1001,1,PEP-L,1,329,EXTRA_CHEESE+40
1,P-1001,2,COKE,2,35,
2,P-1002,1,HAW-L,1,319,
3,P-1002,2,GARLIC,1,89,
4,P-1003,1,PEP-L,1,329,EXTRA_CHEESE+40;NO_CHILI+0
5,P-1004,1,PEP-L,2,349,
6,P-1004,2,COKE,2,35,
7,P-1005,1,PEP-M,1,249,EXTRA_CHEESE+40
8,P-1005,2,HAW-L,1,319,


,orderID,orderTime,itemName,quantity,itemPrice,deliveryFee,merchantCommission
0,GF-201,2026-09-04T11:30:00Z,Pepperoni Pizza (L),1,379,25,146.4
1,GF-201,2026-09-04T11:30:00Z,Garlic Bread,1,109,25,146.4
2,GF-202,2026-09-04T18:10:00Z,Hawaiian Pizza (L),1,369,25,110.7
3,GF-203,2026-09-05T06:00:00Z,Pepperoni Pizza (L),2,399,25,252.9
4,GF-203,2026-09-05T06:00:00Z,Coke,1,45,25,252.9


## 1.2 · Ask one simple question of the sources

The owner asks: **"What was our net revenue by channel on Friday?"** (net = what we keep after the
platform's commission). Here is the honest SQL against the sources:

เจ้าของร้านถามคำถามง่าย ๆ: **"วันศุกร์ได้รายได้สุทธิแต่ละช่องทางเท่าไร?"** (สุทธิ = เงินที่ร้านได้จริงหลังหักค่าคอมมิชชันของแพลตฟอร์ม)
นี่คือ SQL ที่ต้องเขียนจริง ๆ ถ้าถามจาก source ตรง ๆ:

In [3]:
friday_from_sources = q(r'''
WITH pos AS (
    SELECT CASE o.channel WHEN 'QR' THEN 'QR_DINEIN' ELSE o.channel END   AS channel,      -- POS calls it 'QR'
           CAST(o.ordered_at AS TIMESTAMP)                                 AS local_ts,     -- already Bangkok time
           i.qty * (i.unit_price + COALESCE(list_sum(CAST(                                 -- modifier prices are inside
               regexp_extract_all(i.modifiers, '\+(\d+)', 1) AS INT[])), 0))             -- a text column: parse it
                                                                           AS amount,
           0.0                                                             AS commission    -- own channel: no commission
    FROM src.pos_orders o JOIN src.pos_order_items i USING (pos_order_id)
),
grab_orders AS (            -- order fields repeat on every item row, so collapse to one row per order FIRST
    SELECT orderID,
           MAX(timezone('Asia/Bangkok', CAST(orderTime AS TIMESTAMPTZ)))   AS local_ts,     -- UTC -> Bangkok
           SUM(quantity * itemPrice)                                       AS amount,
           MAX(merchantCommission)                                         AS commission    -- MAX, not SUM!
    FROM src.grab_export GROUP BY orderID
),
everything AS (
    SELECT channel, local_ts, amount, commission FROM pos
    UNION ALL
    SELECT 'GRAB', local_ts, amount, commission FROM grab_orders
)
SELECT channel,
       SUM(amount)              AS gross_thb,
       SUM(commission)          AS commission_thb,
       SUM(amount - commission) AS net_thb
FROM everything
WHERE CAST(local_ts - INTERVAL 10 HOUR AS DATE) = DATE '2026-09-04'   -- business day: before 10:00 = previous day
GROUP BY channel ORDER BY channel
''')
friday_from_sources

,channel,gross_thb,commission_thb,net_thb
0,GRAB,857.0,257.1,599.9
1,PHONE,408.0,0.0,408.0
2,QR_DINEIN,808.0,0.0,808.0


It works, but count what that query had to *know*:

| # | Hidden rule the query had to carry | What happens when someone forgets it |
|---|---|---|
| 1 | Two sources, two shapes: union them by hand | A new channel (LINE MAN) means editing every report |
| 2 | Grab is UTC, POS is Bangkok | Grab orders land on the wrong hour and the wrong day |
| 3 | Business day = 10:00–02:00 | The 01:20 order counts towards Saturday |
| 4 | Grab repeats order fields per item: `MAX`, not `SUM` | Commission is double-counted |
| 5 | Modifier prices live inside a text column | Extra-cheese revenue goes missing |
| 6 | `QR` in the POS means `QR_DINEIN` | Channel names don't match across reports |

Every analyst, dashboard and model would re-implement these six rules, slightly differently.
And there's a seventh problem no query can fix:

query นี้ใช้ได้ แต่ลองนับดูว่ามันต้อง *รู้* อะไรบ้าง: ต้องรวมสองแหล่งที่หน้าตาไม่เหมือนกันเอง, Grab เป็น UTC แต่ POS เป็นเวลากรุงเทพ,
วันทำการคือ 10:00–02:00, Grab ซ้ำฟิลด์ออร์เดอร์ทุกแถวจึงต้องใช้ `MAX` ไม่ใช่ `SUM`, ราคา modifier ซ่อนอยู่ในคอลัมน์ text,
และ `QR` ใน POS หมายถึง `QR_DINEIN` ลืมข้อไหนข้อหนึ่ง ตัวเลขก็ผิด
นักวิเคราะห์ทุกคน แดชบอร์ดทุกอัน และโมเดลทุกตัว จะต้องเขียนกฎ 6 ข้อนี้ซ้ำเอง และแต่ละคนก็เขียนไม่เหมือนกัน
แล้วยังมีปัญหาข้อที่ 7 ที่ query ไหนก็แก้ไม่ได้:

In [4]:
# The POS only knows today's price. What was Pepperoni L's list price on Friday? The answer is gone.
# POS รู้แค่ราคาวันนี้ ราคาตั้งของ Pepperoni L เมื่อวันศุกร์คือเท่าไร? คำตอบหายไปแล้ว
q("SELECT * FROM src.pos_menu WHERE item_id = 'PEP-L'")

,item_id,name,price
0,PEP-L,Pepperoni L,349


**7 · History is overwritten.** An operational system answers *"what is true now?"*. Analysis needs
*"what was true then?"*. Once the POS overwrites 329 with 349, "margin before the price rise" cannot
be answered from the POS ever again.

**ประวัติถูกเขียนทับ** ระบบหน้าร้านตอบคำถาม *"ตอนนี้เป็นยังไง"* แต่งานวิเคราะห์ต้องการ *"ตอนนั้นเป็นยังไง"*
พอ POS เขียน 349 ทับ 329 ไปแล้ว คำถาม "margin ก่อนขึ้นราคาเป็นเท่าไร" ก็ตอบจาก POS ไม่ได้อีกเลย

## 1.3 · So what is a warehouse *for*?

| Reason | What it means for the pizza shop | Where it shows up in our design |
|---|---|---|
| **Integrate** | Five sources, one shape. A Grab order and a QR order are both one row in `FCT_ORDER`. | `DIM_CHANNEL`, `BRG_CHANNEL_MENU_ITEM`, decision 01 |
| **Define once** | Business day, net revenue, channel names: one definition, in the load, not in every report. | `date_key` is the business day; `net_revenue_thb` is stored |
| **Keep history** | Prices, recipes, wages change. The warehouse keeps every version. | SCD2 on `DIM_MENU_ITEM`, `BRG_RECIPE`, `DIM_STAFF`; decision 03 |
| **Shape for questions** | Star schema: facts in the middle, context around them. Questions become short `GROUP BY`s. | Every `FCT_*` with `DIM_*` around it |
| **Protect operations** | Heavy queries don't slow down the POS taking a Friday-night order. | Nightly batch from raw, never queries the POS live |

| | Operational DB (POS) | Warehouse |
|---|---|---|
| Built for | Writing one order fast | Reading millions of rows to answer a question |
| Shape | Normalized (3NF), many small tables | Star schema: denormalized dimensions, narrow facts |
| History | Current state, updates in place | Every version kept, facts are append-only |
| Users | Cashier, kitchen screen | Owner, analyst, forecast model |
| Typical query | `WHERE pos_order_id = 'P-1001'` | `GROUP BY business_date, channel` |

สรุปว่า warehouse มีไว้เพื่อ: **รวม** ห้าแหล่งให้เป็นรูปแบบเดียว, **นิยามครั้งเดียว** (วันทำการ รายได้สุทธิ ชื่อช่องทาง ถูกคำนวณตอนโหลด ไม่ใช่ในทุกรายงาน),
**เก็บประวัติ** (ราคา สูตร ค่าแรงเปลี่ยนได้ แต่ warehouse เก็บทุกเวอร์ชัน), **จัดรูปให้ถามง่าย** (star schema: fact อยู่ตรงกลาง บริบทอยู่รอบ ๆ คำถามกลายเป็น `GROUP BY` สั้น ๆ)
และ **ไม่รบกวนระบบหน้าร้าน** (query หนัก ๆ ไม่ทำให้ POS ช้าตอนคืนวันศุกร์ที่ออร์เดอร์เยอะ)

> **Try it.** In the query above, change `MAX(merchantCommission)` to `SUM(...)` and rerun.
> Which channel's net revenue changes, and by exactly how much?
>
> **ลองดู** เปลี่ยน `MAX(merchantCommission)` เป็น `SUM(...)` แล้วรันใหม่ รายได้สุทธิของช่องทางไหนเปลี่ยน และเปลี่ยนไปเท่าไรพอดี?

---
# PART 2 · How we built it (ทำออกมายังไง)

## 2.1 · The architecture, and the four design steps

Chapter 5 describes three warehouse architectures. Ours is the **Kimball dimensional data warehouse**:
every table is organized dimensionally at the lowest level of detail we receive, the stars may be queried
directly, and a **data mart is a subject area inside the warehouse** (Sales, Inventory, Delivery, Customer,
Labour), not a separate database. The `mart_*` tables in the README are *aggregate tables* that serve those data marts.

Every fact table in our README came out of the same four questions, asked in this order:

| Step | Question | Pizza shop example |
|---|---|---|
| **1 · Business process** | What *event* do we want to measure? | Selling an item. Not "sales data" but the act of a customer ordering. |
| **2 · Grain** | What does **one row** mean, exactly? | One menu item on one order (`FCT_ORDER_LINE`) |
| **3 · Dimensions** | What context describes that row? *Who, what, when, where, how?* | Date, channel, customer, menu item |
| **4 · Facts** | What do we *measure* about it? Numbers, at that grain only. | `quantity`, `unit_price_thb`, `line_net_thb` |

The order matters. **Grain comes before columns.** If you start by listing columns, you end up mixing
order-level and line-level numbers in one table (Part 3.1 shows what that does to revenue).

บทที่ 5 พูดถึงสถาปัตยกรรม warehouse สามแบบ ของเราคือ **Kimball dimensional data warehouse**: ทุกตารางจัดแบบ dimensional ที่ระดับละเอียดที่สุดที่เราได้รับ
query จาก star ได้โดยตรง และ **data mart คือ subject area ภายใน warehouse** (Sales, Inventory, Delivery, Customer, Labour) ไม่ใช่ฐานข้อมูลแยก
ส่วนตาราง `mart_*` ใน README คือ *aggregate table* ที่ให้บริการ data mart เหล่านั้น

fact ทุกตัวใน README มาจาก 4 คำถามเดียวกัน ถามตามลำดับนี้:
1. **Business process** อยากวัด *เหตุการณ์* อะไร เช่น การขายเมนูหนึ่งรายการ ไม่ใช่ "ข้อมูลการขาย" ลอย ๆ แต่คือการที่ลูกค้าสั่ง
2. **Grain** ข้อมูล **หนึ่งแถว** หมายถึงอะไรกันแน่ เช่น เมนูหนึ่งรายการในออร์เดอร์หนึ่งใบ
3. **Dimensions** บริบทอะไรอธิบายแถวนั้น *ใคร อะไร เมื่อไร ที่ไหน ยังไง* เช่น วันที่ ช่องทาง ลูกค้า เมนู
4. **Facts** เราวัด *ตัวเลข* อะไรของมัน และต้องเป็นตัวเลขที่ grain นั้นเท่านั้น

ลำดับสำคัญมาก **ต้องกำหนด grain ก่อนเลือกคอลัมน์** ถ้าเริ่มจากการไล่คอลัมน์ จะได้ตารางที่ปนตัวเลขระดับออร์เดอร์กับระดับบรรทัดเข้าด้วยกัน (Part 3.1 จะโชว์ว่ามันทำให้รายได้เพี้ยนยังไง)

## 2.2 · Step 1 for the whole shop: the conformance matrix

Doing step 1 for the whole shop gives a list of fact tables. Putting them against the dimensions, and against
each dimension's **hierarchy levels**, gives the **conformance matrix**: the one-page plan of the warehouse.

In [5]:
# Fact tables (rows) x dimension hierarchy levels (columns), as in README section 11.
# fact table (แถว) x ระดับ hierarchy ของ dimension (คอลัมน์) ตาม README section 11
levels = ["day", "month", "quarter", "channel", "channel type", "customer", "SKU", "category",
          "modifier", "promotion", "ingredient", "ingr. category", "supplier", "staff", "role", "order_number"]
D, C, K, M, X, P, I, U, S, N = (["day", "month", "quarter"], ["channel", "channel type"], ["customer"],
                                ["SKU", "category"], ["modifier"], ["promotion"], ["ingredient", "ingr. category"],
                                ["supplier"], ["staff", "role"], ["order_number"])
conformance = {
    "FCT_ORDER":                 D + C + K + N,
    "FCT_ORDER_LINE":            D + C + K + M + N,
    "FCT_ORDER_LINE_MODIFIER":   D + C + M + X + N,
    "FCT_PAYMENT":               D + C + K + N,
    "FCT_ORDER_PROMOTION":       D + C + P + N,
    "FCT_ORDER_LINE_PROMOTION":  D + C + K + M + P + N,
    "FCT_ORDER_EVENT":           D + C + S + N,
    "FCT_PURCHASE_ORDER(_LINE)": D + I + U,
    "FCT_INVENTORY_MOVEMENT":    D + M + X + I + U,
    "FCT_INVENTORY_COUNT":       D + I + S,
    "FCT_SHIFT":                 D + S,
    "fct_forecast":              D + C + M,
}
matrix = pd.DataFrame([[("✓" if lv in used else "") for lv in levels] for used in conformance.values()],
                      index=list(conformance.keys()), columns=levels)
matrix.loc["fact tables conforming"] = [sum(lv in used for used in conformance.values()) for lv in levels]
matrix

,day,month,quarter,channel,channel type,customer,SKU,category,modifier,promotion,ingredient,ingr. category,supplier,staff,role,order_number
FCT_ORDER,✓,✓,✓,✓,✓,✓,,,,,,,,,,✓
FCT_ORDER_LINE,✓,✓,✓,✓,✓,✓,✓,✓,,,,,,,,✓
FCT_ORDER_LINE_MODIFIER,✓,✓,✓,✓,✓,,✓,✓,✓,,,,,,,✓
FCT_PAYMENT,✓,✓,✓,✓,✓,✓,,,,,,,,,,✓
FCT_ORDER_PROMOTION,✓,✓,✓,✓,✓,,,,,✓,,,,,,✓
FCT_ORDER_LINE_PROMOTION,✓,✓,✓,✓,✓,✓,✓,✓,,✓,,,,,,✓
FCT_ORDER_EVENT,✓,✓,✓,✓,✓,,,,,,,,,✓,✓,✓
FCT_PURCHASE_ORDER(_LINE),✓,✓,✓,,,,,,,,✓,✓,✓,,,
FCT_INVENTORY_MOVEMENT,✓,✓,✓,,,,✓,✓,✓,,✓,✓,✓,,,
FCT_INVENTORY_COUNT,✓,✓,✓,,,,,,,,✓,✓,,✓,✓,


**How to read it.** Each row is a fact table. Each column is a level of a dimension that is built **once** and
shared: a **conformed dimension**. The last row shows the payoff:

- **Date is conformed by every fact table.** One `DIM_DATE`, keyed on the *business* day, with day → month → quarter → year.
  Sales, stock, shifts and purchases can be compared by day or by month without anyone redefining "Friday".
- **Menu item is shared by sales and stock.** That's the bridge for the closed loop in README §9:
  units sold × `BRG_RECIPE` = grams of cheese used.
- **Staff is shared by delivery, counting and labour.** So "who was on shift when tickets ran late" is a drill-across, not a project.
- **`order_number` is a degenerate dimension** shared by the order facts, which is how they are compared with each other.

Without conformance, the warehouse becomes *islands of information* (slide 54). **Conformed dimensions are what make separate fact tables one warehouse.**

**วิธีอ่าน** แต่ละแถวคือ fact table หนึ่งตัว แต่ละคอลัมน์คือระดับหนึ่งของ dimension ที่สร้าง **ครั้งเดียว** แล้วใช้ร่วมกัน เรียกว่า **conformed dimension**
**Date ถูกใช้โดย fact ทุกตัว** มี `DIM_DATE` ตัวเดียวที่ใช้ *วันทำการ* เป็น key และมี hierarchy วัน → เดือน → ไตรมาส → ปี,
**Menu item ใช้ร่วมกันระหว่างการขายกับสต็อก** คือสะพานของ closed loop ใน README §9, **Staff ใช้ร่วมกันใน delivery, การนับสต็อก และกะงาน**
และ **`order_number` เป็น degenerate dimension** ที่ fact ฝั่งออร์เดอร์ใช้ร่วมกัน จึงเทียบกันได้
ถ้าไม่มี conformance warehouse จะกลายเป็น *เกาะข้อมูล* (สไลด์ 54) **conformed dimension คือสิ่งที่ทำให้ fact หลายตัวกลายเป็น warehouse เดียวกัน**

## 2.3 · Build the dimensions

A dimension answers *who / what / when / where / how*. Six chapter 5 conventions shape ours:

1. **Surrogate keys ending in `_key`** (`menu_item_key`, a plain integer the warehouse assigns) instead of the source's ID.
   Natural keys end in `_nk`. Pepperoni L needs *two* rows (329 and 349), so `PEP-L` cannot be the primary key, and
   source IDs collide across systems.
2. **SCD Type 2** for things whose history matters: a change adds a new row with `valid_from` / `valid_to`
   instead of overwriting. Here `valid_to` is *exclusive* (the first day the row is no longer true).
3. **Special-case rows.** Every dimension has key 0 = *Not applicable*, 1 = *Unknown*, 2 = *Invalid*, so a fact never needs a NULL key (Part 3.5).
4. **Flags hold readable values** ("Own channel" / "Aggregator", "Weekend" / "Weekday"), and **codes travel with descriptions** (`size_code` L + `size_name` Large).
5. **Attribute hierarchies** for drilling: day → month → quarter → year; SKU → category; channel → channel type.
6. **No snowflakes.** `category` and `size_code` sit right on `DIM_MENU_ITEM`; role sits on `DIM_STAFF`. One join from a fact to its context.

dimension ตอบคำถาม *ใคร / อะไร / เมื่อไร / ที่ไหน / ยังไง* ข้อตกลงจากบทที่ 5 ที่ใช้มีหกข้อ:
1. **Surrogate key ลงท้ายด้วย `_key`** (เช่น `menu_item_key` เลขจำนวนเต็มที่ warehouse ออกให้เอง) natural key ลงท้ายด้วย `_nk` เพราะ Pepperoni L ต้องมี *สองแถว* และ ID จากต่างระบบชนกันได้
2. **SCD Type 2** สำหรับสิ่งที่ประวัติสำคัญ: เปลี่ยนแล้วเพิ่มแถวใหม่ที่มี `valid_from` / `valid_to` แทนการเขียนทับ (`valid_to` เป็นแบบ *exclusive*)
3. **Special-case row** ทุก dimension มี key 0 = *ไม่เกี่ยวข้อง*, 1 = *ไม่ทราบ*, 2 = *ไม่ถูกต้อง* fact จึงไม่ต้องมี key ที่เป็น NULL เลย (Part 3.5)
4. **Flag เก็บค่าที่อ่านออก** ("Own channel" / "Aggregator") และ **รหัสมาพร้อมคำอธิบาย** (`size_code` L + `size_name` Large)
5. **Attribute hierarchy** สำหรับ drill: วัน → เดือน → ไตรมาส → ปี, SKU → หมวด, ช่องทาง → ประเภทช่องทาง
6. **ไม่มี snowflake** `category` และ `size_code` อยู่ใน `DIM_MENU_ITEM` เลย role อยู่ใน `DIM_STAFF` join จาก fact ไปหาบริบทแค่ครั้งเดียว

In [6]:
# Special-case rows every dimension starts with  |  special-case row ที่ทุก dimension มี
SPECIAL = [(0, "Not applicable"), (1, "Unknown"), (2, "Invalid")]

# DIM_DATE: one row per *business* day, with the day > month > quarter > year hierarchy
# DIM_DATE: หนึ่งแถวต่อหนึ่ง *วันทำการ* พร้อม hierarchy วัน > เดือน > ไตรมาส > ปี
con.execute('''
CREATE TABLE dw.dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INT)                      AS date_key,
       CAST(d AS DATE)                                         AS full_date,
       dayname(d)                                              AS day_name,
       CASE WHEN isodow(d) IN (6, 7) THEN 'Weekend' ELSE 'Weekday' END AS weekend_flag,
       'Not a holiday'                                         AS holiday_flag,
       monthname(d)                                            AS month_name,
       'Q' || quarter(d) || ' ' || year(d)                     AS quarter,
       year(d)                                                 AS year,
       'Day'                                                   AS row_type
FROM generate_series(DATE '2026-09-01', DATE '2026-09-30', INTERVAL 1 DAY) AS t(d)
UNION ALL
SELECT k, NULL, 'n/a', 'n/a', 'n/a', 'n/a', 'n/a', NULL, label FROM (VALUES (0, 'Not applicable'), (1, 'Unknown')) AS s(k, label)
''')

def with_special(rows, columns, fill):
    # prepend the three special-case rows; `fill` gives their value for each non-key column
    # เพิ่ม special-case row สามแถวไว้ข้างหน้า `fill` คือค่าของคอลัมน์อื่นในแถวพิเศษ
    special = [(k, *[fill.get(c, "n/a") for c in columns[1:-1]], label) for k, label in SPECIAL]
    return pd.DataFrame(special + [(*r, columns[-1] == "row_type" and "Member") for r in rows], columns=columns)

load("dw.dim_channel", with_special([
    (101, "QR_DINEIN", "QR dine-in", "Dine-in",  "Own channel",  0.0),
    (102, "PHONE",     "Phone",      "Dine-in",  "Own channel",  0.0),
    (103, "GRAB",      "Grab",       "Delivery", "Aggregator",  30.0),
    (104, "LINEMAN",   "LINE MAN",   "Delivery", "Aggregator",  30.0),   # no orders yet, but the dimension is ready
], ["channel_key", "channel_code", "channel_name", "channel_type", "channel_ownership", "commission_rate_pct", "row_type"],
   {"commission_rate_pct": 0.0}).assign(row_type=lambda d: d.row_type.replace("Member", "Channel")))

load("dw.dim_customer", with_special([
    (101, "m-081", "Member", "Regular"), (102, "m-092", "Member", "New"),   # own channels only  |  เฉพาะช่องทางของร้าน
], ["customer_key", "customer_nk", "membership", "rfm_segment", "row_type"], {}).assign(
    row_type=lambda d: d.row_type.replace("Member", "Customer")))

# DIM_MENU_ITEM is SCD2: Pepperoni L has two rows, one per price  |  DIM_MENU_ITEM เป็น SCD2: Pepperoni L มีสองแถว
far, start, rise = pd.Timestamp("9999-12-31"), pd.Timestamp("2026-01-01"), pd.Timestamp("2026-09-05")
menu = with_special([
    (101, "PEP-M",  "Pepperoni M",  "pizza", "M", "Medium", 249, start, far,  "Current"),
    (102, "PEP-L",  "Pepperoni L",  "pizza", "L", "Large",  329, start, rise, "Expired"),
    (103, "HAW-L",  "Hawaiian L",   "pizza", "L", "Large",  319, start, far,  "Current"),
    (104, "GARLIC", "Garlic Bread", "side",  "-", "n/a",     89, start, far,  "Current"),
    (105, "COKE",   "Coke",         "drink", "-", "n/a",     35, start, far,  "Current"),
    (106, "PEP-L",  "Pepperoni L",  "pizza", "L", "Large",  349, rise,  far,  "Current"),
], ["menu_item_key", "menu_item_nk", "item_name", "category", "size_code", "size_name", "base_price_thb",
    "valid_from", "valid_to", "current_flag", "row_type"],
   {"base_price_thb": None, "valid_from": start, "valid_to": far, "current_flag": "Current"})
load("dw.dim_menu_item", menu.assign(row_type=menu.row_type.replace("Member", "Menu item")))

load("dw.dim_modifier", with_special([
    (101, "EXTRA_CHEESE", "extra cheese", "add",    40),
    (102, "NO_CHILI",     "no chili",     "remove",  0),
], ["modifier_key", "modifier_nk", "modifier_name", "modifier_type", "default_price_delta_thb", "row_type"],
   {"default_price_delta_thb": 0}).assign(row_type=lambda d: d.row_type.replace("Member", "Modifier")))

# Junk dimension for the order's status codes  |  junk dimension สำหรับรหัสสถานะของออร์เดอร์
load("dw.dim_order_info", pd.DataFrame([
    (101, "completed", "Not cancelled"), (102, "cancelled", "Cancelled"), (103, "refunded", "Not cancelled"),
], columns=["order_info_key", "order_status", "cancellation_flag"]))

# How each channel lists each item (points at menu_item_nk so it survives SCD2 versions)
# แต่ละช่องทางลงขายเมนูไหนด้วยชื่ออะไร (ชี้ไปที่ menu_item_nk เพื่อให้ใช้ได้ทุกเวอร์ชันของ SCD2)
load("dw.brg_channel_menu_item", pd.DataFrame(
    [(101, nk, nk, None, "Listed") for nk in ["PEP-M", "PEP-L", "HAW-L", "GARLIC", "COKE"]] +
    [(103, "PEP-M",  "Pepperoni Pizza (M)", 289, "Listed"),
     (103, "PEP-L",  "Pepperoni Pizza (L)", 399, "Listed"),
     (103, "HAW-L",  "Hawaiian Pizza (L)",  369, "Listed"),
     (103, "GARLIC", "Garlic Bread",        109, "Listed"),
     (103, "COKE",   "Coke",                 45, "Listed")],
    columns=["channel_key", "menu_item_nk", "platform_item_name", "channel_price_thb", "listing_status"]))

load("dw.dim_ingredient", with_special([
    (101, "mozzarella", "dairy",    "gram"),
    (102, "pepperoni",  "protein",  "gram"),
    (103, "chili",      "dry_good", "gram"),
], ["ingredient_key", "ingredient_nk", "category", "base_uom", "row_type"], {}).assign(
    row_type=lambda d: d.row_type.replace("Member", "Ingredient")))

load("dw.dim_supplier", with_special([(101, "SUP-01", "Bangkok Dairy Co.", 1)],
    ["supplier_key", "supplier_nk", "supplier_name", "lead_time_days", "row_type"], {"lead_time_days": 0}).assign(
    row_type=lambda d: d.row_type.replace("Member", "Supplier")))

load("dw.brg_recipe", pd.DataFrame([            # grams of each ingredient per 1 unit sold  |  กรัมของวัตถุดิบต่อการขาย 1 ชิ้น
    ("PEP-M", 101, 100), ("PEP-M", 102, 40), ("PEP-M", 103, 5),
    ("PEP-L", 101, 150), ("PEP-L", 102, 60), ("PEP-L", 103, 5),
    ("HAW-L", 101, 150),
    ("GARLIC", 101, 20),
], columns=["menu_item_nk", "ingredient_key", "qty_per_unit"]))

load("dw.brg_modifier_recipe", pd.DataFrame([   # signed: extra cheese uses more, no chili uses less  |  มีเครื่องหมาย
    (101, 101, +40),
    (102, 103,  -5),
], columns=["modifier_key", "ingredient_key", "qty_delta_per_unit"]))

q("SELECT * FROM dw.dim_menu_item ORDER BY menu_item_key")

,menu_item_key,menu_item_nk,item_name,category,size_code,size_name,base_price_thb,valid_from,valid_to,current_flag,row_type
0,0,n/a,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Current,Not applicable
1,1,n/a,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Current,Unknown
2,2,n/a,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Current,Invalid
3,101,PEP-M,Pepperoni M,pizza,M,Medium,249.0,2026-01-01,9999-12-31,Current,Menu item
4,102,PEP-L,Pepperoni L,pizza,L,Large,329.0,2026-01-01,2026-09-05,Expired,Menu item
5,103,HAW-L,Hawaiian L,pizza,L,Large,319.0,2026-01-01,9999-12-31,Current,Menu item
6,104,GARLIC,Garlic Bread,side,-,n/a,89.0,2026-01-01,9999-12-31,Current,Menu item
7,105,COKE,Coke,drink,-,n/a,35.0,2026-01-01,9999-12-31,Current,Menu item
8,106,PEP-L,Pepperoni L,pizza,L,Large,349.0,2026-09-05,9999-12-31,Current,Menu item


Keys 0, 1 and 2 are the special-case rows. Pepperoni L is two rows, 102 (329, expired) and 106 (349, current),
and both share the natural key `PEP-L`.

key 0, 1 และ 2 คือ special-case row ส่วน Pepperoni L มีสองแถว คือ 102 (329, หมดอายุแล้ว) และ 106 (349, ปัจจุบัน) โดยใช้ natural key `PEP-L` ร่วมกัน

## 2.4 · Staging: one common shape

Before anything goes into a fact table, every source is turned into the **same shape**. This is
the trusted-zone work from notebook 1, compressed into three SQL statements: one order table, one
line table, one modifier table, whatever the source.

ก่อนจะเข้า fact table ทุกแหล่งต้องถูกแปลงเป็น **รูปแบบเดียวกัน** ก่อน นี่คืองานใน trusted zone ของโน้ตบุ๊กแรก ย่อเหลือ SQL สามคำสั่ง:
ตารางออร์เดอร์หนึ่งตาราง ตารางบรรทัดหนึ่งตาราง และตาราง modifier หนึ่งตาราง ไม่ว่าข้อมูลจะมาจากแหล่งไหน

In [7]:
con.execute('''
CREATE TABLE stg.order_header AS
SELECT CASE channel WHEN 'QR' THEN 'QR_DINEIN' ELSE channel END AS channel_code,
       pos_order_id                    AS order_number,
       CAST(ordered_at AS TIMESTAMP)   AS order_ts,
       member_id                       AS customer_nk,
       0.0                             AS delivery_fee_thb,
       0.0                             AS commission_amount_thb
FROM src.pos_orders
UNION ALL
SELECT 'GRAB', orderID,
       MAX(timezone('Asia/Bangkok', CAST(orderTime AS TIMESTAMPTZ))),   -- UTC -> Bangkok, once, here
       NULL,                                                            -- aggregators are anonymous
       MAX(deliveryFee), MAX(merchantCommission)                        -- repeated per item: take it once
FROM src.grab_export GROUP BY orderID
''')

con.execute('''
CREATE TABLE stg.order_line AS
SELECT CASE o.channel WHEN 'QR' THEN 'QR_DINEIN' ELSE o.channel END AS channel_code,
       i.pos_order_id AS order_number, i.line_no AS line_number, i.item_id AS menu_item_nk,
       i.qty AS quantity, i.unit_price AS unit_price_thb
FROM src.pos_order_items i JOIN src.pos_orders o USING (pos_order_id)
UNION ALL
SELECT 'GRAB', g.orderID,
       row_number() OVER (PARTITION BY g.orderID ORDER BY g.itemName),
       m.menu_item_nk,                                                  -- Grab's name -> our item, via the bridge
       g.quantity, g.itemPrice
FROM src.grab_export g
LEFT JOIN dw.brg_channel_menu_item m ON m.channel_key = 103 AND m.platform_item_name = g.itemName
''')

con.execute('''
CREATE TABLE stg.line_modifier AS               -- "EXTRA_CHEESE+40;NO_CHILI+0" -> one row each
SELECT h.channel_code, s.order_number, s.line_number,
       split_part(s.m, '+', 1)                          AS modifier_nk,
       1                                                AS quantity,
       CAST(split_part(s.m, '+', 2) AS DECIMAL(10, 2))  AS price_delta_thb
FROM (SELECT pos_order_id AS order_number, line_no AS line_number, unnest(string_split(modifiers, ';')) AS m
      FROM src.pos_order_items WHERE modifiers <> '') s
JOIN stg.order_header h USING (order_number)
''')

display(q("SELECT * FROM stg.order_line ORDER BY order_number, line_number"))
display(q("SELECT * FROM stg.line_modifier"))

,channel_code,order_number,line_number,menu_item_nk,quantity,unit_price_thb
0,GRAB,GF-201,1,GARLIC,1,109
1,GRAB,GF-201,2,PEP-L,1,379
2,GRAB,GF-202,1,HAW-L,1,369
3,GRAB,GF-203,1,COKE,1,45
4,GRAB,GF-203,2,PEP-L,2,399
5,QR_DINEIN,P-1001,1,PEP-L,1,329
6,QR_DINEIN,P-1001,2,COKE,2,35
7,PHONE,P-1002,1,HAW-L,1,319
8,PHONE,P-1002,2,GARLIC,1,89
9,QR_DINEIN,P-1003,1,PEP-L,1,329


,channel_code,order_number,line_number,modifier_nk,quantity,price_delta_thb
0,QR_DINEIN,P-1001,1,EXTRA_CHEESE,1,40.0
1,QR_DINEIN,P-1003,1,NO_CHILI,1,0.0
2,PHONE,P-1005,1,EXTRA_CHEESE,1,40.0
3,QR_DINEIN,P-1003,1,EXTRA_CHEESE,1,40.0


## 2.5 · Load the facts: the surrogate key pipeline

Loading a fact is always the same move: **take a staged row, swap every business key for the
dimension's surrogate key, keep the measures.** Every fact gets *its own* key to each dimension it is
analysed by, so the line fact carries date, channel and customer itself instead of borrowing them from the order fact.

- `channel_code` → `channel_key`: a plain lookup.
- `order_ts` → `date_key`: the **business-day rule lives here, once**. Nobody downstream ever subtracts 10 hours again.
- `customer_nk` → `customer_key`: Grab orders and walk-in non-members have no identified customer, so they get **row 0, Not applicable**, never NULL.
- `menu_item_nk` → `menu_item_key`: an **as-of join**. The *version* of Pepperoni L that was true on that business day.
  Friday lines get key 102 (329), Saturday lines get key 106 (349). An item nobody has mapped yet gets **row 1, Unknown**.

Then a **quality check**: count the rows that landed on Unknown and raise an alert. The row is *loaded*, so
Friday's sales are not understated (slide 72), and it is re-pointed once the item is mapped.

การโหลด fact ทำแบบเดิมทุกครั้ง: **เอาแถวจาก staging มา เปลี่ยน business key ทุกตัวเป็น surrogate key ของ dimension แล้วเก็บตัวเลขไว้**
fact ทุกตัวได้ key ของ *ตัวเอง* ไปหาทุก dimension ที่ใช้วิเคราะห์ fact ระดับบรรทัดจึงมีวันที่ ช่องทาง และลูกค้าเอง ไม่ได้ยืมจาก fact ระดับออร์เดอร์
`channel_code` → `channel_key` คือ lookup ธรรมดา, `order_ts` → `date_key` คือ **ที่ที่กฎวันทำการอยู่ ครั้งเดียว**,
`customer_nk` → `customer_key` ออร์เดอร์ Grab ไม่มีลูกค้า จึงได้ **แถว 0 ไม่เกี่ยวข้อง** ไม่ใช่ NULL,
`menu_item_nk` → `menu_item_key` คือ **as-of join** บรรทัดวันศุกร์ได้ key 102 (329) บรรทัดวันเสาร์ได้ key 106 (349) ส่วนเมนูที่ยังไม่ได้ map จะได้ **แถว 1 ไม่ทราบ**
จากนั้นคือ **การตรวจคุณภาพ**: นับแถวที่ไปตกที่ Unknown แล้วแจ้งเตือน แถวนั้นยัง *ถูกโหลด* ยอดขายวันศุกร์จึงไม่ต่ำกว่าความจริง (สไลด์ 72) และจะถูกชี้ใหม่เมื่อ map เมนูแล้ว

In [8]:
# FCT_ORDER: grain = one row per order, any channel  |  grain = หนึ่งแถวต่อหนึ่งออร์เดอร์ ทุกช่องทาง
con.execute('''
CREATE TABLE stg.order_keys AS                       -- the dimension keys every order-level fact will reuse
SELECT h.*, c.channel_key,
       CAST(strftime(h.order_ts - INTERVAL 10 HOUR, '%Y%m%d') AS INT) AS date_key,   -- business-day rule, ONCE
       COALESCE(cu.customer_key, 0)                                  AS customer_key, -- 0 = Not applicable
       101                                                           AS order_info_key  -- completed
FROM stg.order_header h
JOIN dw.dim_channel c ON c.channel_code = h.channel_code
LEFT JOIN dw.dim_customer cu ON cu.customer_nk = h.customer_nk
''')
con.execute('''
CREATE TABLE dw.fct_order AS
WITH line_totals AS (SELECT channel_code, order_number, SUM(quantity * unit_price_thb) AS items_thb FROM stg.order_line GROUP BY ALL),
     mod_totals  AS (SELECT channel_code, order_number, SUM(quantity * price_delta_thb) AS mods_thb FROM stg.line_modifier GROUP BY ALL)
SELECT k.date_key, k.channel_key, k.customer_key, k.order_info_key,
       k.order_number,                                                             -- degenerate dimension
       k.order_ts,
       lt.items_thb + COALESCE(mt.mods_thb, 0)                           AS gross_amount_thb,
       0.0                                                               AS discount_amount_thb,
       k.delivery_fee_thb,
       k.commission_amount_thb,
       lt.items_thb + COALESCE(mt.mods_thb, 0) - k.commission_amount_thb AS net_revenue_thb
FROM stg.order_keys k
JOIN line_totals lt     ON lt.channel_code = k.channel_code AND lt.order_number = k.order_number
LEFT JOIN mod_totals mt ON mt.channel_code = k.channel_code AND mt.order_number = k.order_number
''')

# FCT_ORDER_LINE: grain = one menu item on one order; carries its OWN date, channel and customer keys
# FCT_ORDER_LINE: grain = เมนูหนึ่งรายการในออร์เดอร์หนึ่งใบ มี key วันที่ ช่องทาง ลูกค้า เป็นของตัวเอง
con.execute('''
CREATE TABLE dw.fct_order_line AS
SELECT k.date_key, k.channel_key, k.customer_key,
       COALESCE(mi.menu_item_key, 1)                       AS menu_item_key,  -- 1 = Unknown, never NULL
       k.order_info_key,
       l.order_number, l.line_number,                                         -- degenerate dimensions
       l.quantity,
       l.unit_price_thb,                                                      -- what the customer actually paid
       0.0                                                 AS line_discount_thb,
       l.quantity * l.unit_price_thb                       AS line_net_thb
FROM stg.order_line l
JOIN stg.order_keys k ON k.channel_code = l.channel_code AND k.order_number = l.order_number
JOIN dw.dim_date d    ON d.date_key = k.date_key
LEFT JOIN dw.dim_menu_item mi
       ON mi.menu_item_nk = l.menu_item_nk AND mi.row_type = 'Menu item'
      AND d.full_date >= mi.valid_from AND d.full_date < mi.valid_to          -- as-of join (SCD2)
''')

# FCT_ORDER_LINE_MODIFIER: grain = one modifier on one order line  |  grain = modifier หนึ่งตัวบนบรรทัดหนึ่งบรรทัด
con.execute('''
CREATE TABLE dw.fct_order_line_modifier AS
SELECT l.date_key, l.channel_key, l.menu_item_key, COALESCE(md.modifier_key, 1) AS modifier_key,
       s.order_number, s.line_number, s.quantity, s.price_delta_thb
FROM stg.line_modifier s
JOIN dw.fct_order_line l ON l.order_number = s.order_number AND l.line_number = s.line_number
LEFT JOIN dw.dim_modifier md ON md.modifier_nk = s.modifier_nk
''')

# Quality check: no NULL keys anywhere, and alert on rows parked on Unknown  |  ตรวจว่าไม่มี key ที่เป็น NULL และแจ้งเตือนแถวที่ไปตกที่ Unknown
nulls = con.execute('''SELECT count(*) FROM dw.fct_order_line
                       WHERE date_key IS NULL OR channel_key IS NULL OR customer_key IS NULL OR menu_item_key IS NULL''').fetchone()[0]
unknown = con.execute("SELECT count(*) FROM dw.fct_order_line WHERE menu_item_key = 1").fetchone()[0]
assert nulls == 0, "a fact key is NULL - the load is broken"
print(f"no NULL keys; {unknown} line(s) parked on menu_item_key 1 (Unknown)" + (" -> ALERT: map the item" if unknown else ""))

q('''
SELECT l.order_number, d.full_date AS business_date, l.line_number, l.channel_key, l.customer_key,
       l.menu_item_key, mi.item_name, mi.base_price_thb AS list_price_that_day, l.unit_price_thb AS paid
FROM dw.fct_order_line l
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_menu_item mi USING (menu_item_key)
WHERE mi.menu_item_nk = 'PEP-L' ORDER BY d.full_date, l.order_number
''')

no NULL keys; 0 line(s) parked on menu_item_key 1 (Unknown)


,order_number,business_date,line_number,channel_key,customer_key,menu_item_key,item_name,list_price_that_day,paid
0,GF-201,2026-09-04,2,103,0,102,Pepperoni L,329.0,379
1,P-1001,2026-09-04,1,101,101,102,Pepperoni L,329.0,329
2,P-1003,2026-09-04,1,101,0,102,Pepperoni L,329.0,329
3,GF-203,2026-09-05,2,103,0,106,Pepperoni L,349.0,399
4,P-1004,2026-09-05,1,101,0,106,Pepperoni L,349.0,349


Look at **P-1003**: ordered at 01:20 on the 5th, business date the 4th, menu key 102 (329). **GF-201** paid 379
while the list price was 329, because Grab prices are uplifted, and its `customer_key` is 0: an aggregator order
has no customer, and the fact says so explicitly instead of holding a NULL.

ดู **P-1003**: สั่งตอนตี 1:20 ของวันที่ 5 แต่วันทำการคือวันที่ 4 และได้ menu key 102 (329)
ส่วน **GF-201** จ่าย 379 ขณะที่ราคาตั้งคือ 329 เพราะราคาบน Grab ถูกบวกเพิ่ม และมี `customer_key` เป็น 0 คือออร์เดอร์จากแพลตฟอร์มไม่มีลูกค้า fact บอกตรง ๆ แทนที่จะเก็บ NULL

## 2.6 · The same question, asked of the warehouse

In [9]:
friday_from_warehouse = q('''
SELECT c.channel_code AS channel,
       SUM(o.gross_amount_thb)      AS gross_thb,
       SUM(o.commission_amount_thb) AS commission_thb,
       SUM(o.net_revenue_thb)       AS net_thb
FROM dw.fct_order o
JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_date d    USING (date_key)
WHERE d.full_date = DATE '2026-09-04'
GROUP BY ALL ORDER BY channel
''')
display(friday_from_warehouse)

same = friday_from_sources.astype({"gross_thb": float}).round(2).equals(friday_from_warehouse.astype({"gross_thb": float}).round(2))
print("same answer as the 30-line source query:", same)

,channel,gross_thb,commission_thb,net_thb
0,GRAB,857.0,257.1,599.9
1,PHONE,408.0,0.0,408.0
2,QR_DINEIN,808.0,0.0,808.0


same answer as the 30-line source query: True


Same numbers, and the query has **no rules in it at all**: no timezone, no business-day arithmetic,
no `MAX` trick, no text parsing. Those were all paid for **once**, at load time. That's the whole
argument for a warehouse in one cell.

ตัวเลขเท่ากันเป๊ะ และ query นี้ **ไม่มีกฎธุรกิจอยู่ข้างในเลย** ไม่มี timezone ไม่มีการคำนวณวันทำการ ไม่มีทริก `MAX` ไม่มีการ parse text
ทั้งหมดถูกจ่ายไป **ครั้งเดียว** ตอนโหลด นี่คือเหตุผลทั้งหมดของการทำ warehouse ในเซลล์เดียว

> **Try it.** Add a row to `grab_export` with `itemName = "Pepperoni Pizza (XL)"` (not in the bridge) and rerun from 1.1.
> The line loads against `menu_item_key` 1 and the check raises an alert. Compare Friday's Grab revenue with and
> without that row: why is "load it as Unknown" better than holding it back until someone maps the item?
>
> **ลองดู** เพิ่มแถวใน `grab_export` ที่ `itemName = "Pepperoni Pizza (XL)"` (ไม่มีใน bridge) แล้วรันใหม่ตั้งแต่ 1.1 บรรทัดนั้นจะถูกโหลดเข้า `menu_item_key` 1 และมีการแจ้งเตือน
> เทียบรายได้ Grab วันศุกร์ตอนมีและไม่มีแถวนั้น ทำไม "โหลดเป็น Unknown" ถึงดีกว่าการกักไว้จนกว่าจะมีคน map เมนู?

---
# PART 3 · Designing fact tables (ออกแบบ Fact Table)

A **fact table** records one kind of business event. Every row has three kinds of column:

```
FCT_ORDER_LINE
├── surrogate keys      date_key, channel_key, customer_key,   → who / what / when / where (via dimensions)
│                       menu_item_key, order_info_key
├── degenerate dims     order_number, line_number              → transaction identifiers with no table of their own
└── facts (measures)    quantity, unit_price_thb,              → the numbers, true at THIS grain
                        line_net_thb
```

Facts are **deep** (few columns, many rows, append-only). Dimensions are **wide** (many descriptive columns,
few rows). A fact table holds a key to *every* dimension it is analysed by, and never a key to another fact table.
Our model has 13 fact tables. The rules below are the reasons each one looks the way it does.

**Fact table** บันทึกเหตุการณ์ทางธุรกิจหนึ่งประเภท ทุกแถวมีคอลัมน์สามแบบ: **surrogate key** ไปหาบริบทผ่าน dimension,
**degenerate dimension** (รหัสธุรกรรมที่ไม่มีตารางของตัวเอง เช่น เลขออร์เดอร์ เลขบรรทัด) และ **fact** (ตัวเลขที่ถูกต้องที่ grain นี้)
fact จะ **ลึก** (คอลัมน์น้อย แถวเยอะ เพิ่มอย่างเดียว) ส่วน dimension จะ **กว้าง** (คอลัมน์บรรยายเยอะ แถวน้อย)
fact table มี key ไปหา *ทุก* dimension ที่ใช้วิเคราะห์ และไม่มี key ไปหา fact table ตัวอื่นเลย
โมเดลของเรามี fact 13 ตัว กฎข้างล่างคือเหตุผลที่แต่ละตัวหน้าตาเป็นแบบนั้น

## 3.1 · Rule 1: declare the grain, never mix two grains, and drill across

**Why `FCT_ORDER` and `FCT_ORDER_LINE` are two tables.** Commission and delivery fee belong to the *order*.
Quantity and price belong to the *line*. Facts with different grain are different processes and get different
fact tables (slide 44). If commission sat on the line table, it would repeat on every line of the order, exactly
like the Grab export does, and any `SUM` would multiply it.

The same trap catches you when you **join two fact tables** and sum the one with the coarser grain (slide 49):

**ทำไม `FCT_ORDER` กับ `FCT_ORDER_LINE` ถึงเป็นสองตาราง** ค่าคอมมิชชันกับค่าส่งเป็นของ *ออร์เดอร์* ส่วนจำนวนและราคาเป็นของ *บรรทัด*
fact ที่ grain ต่างกันคือคนละ process และต้องอยู่คนละ fact table (สไลด์ 44) ถ้าเอาค่าคอมมิชชันไปไว้ในตารางบรรทัด มันจะซ้ำในทุกบรรทัด เหมือนไฟล์ Grab และ `SUM` จะได้ค่าคูณเพิ่ม
กับดักเดียวกันเกิดตอน **join fact สองตัวเข้าด้วยกัน** แล้ว sum ตัวที่ grain หยาบกว่า (สไลด์ 49):

In [10]:
q('''
SELECT 'right: order grain only'                AS how, SUM(commission_amount_thb) AS grab_commission_friday
FROM dw.fct_order WHERE channel_key = 103 AND date_key = 20260904
UNION ALL
SELECT 'wrong: two facts joined, then summed',  SUM(o.commission_amount_thb)
FROM dw.fct_order o JOIN dw.fct_order_line l USING (order_number)      -- GF-201 has 2 lines -> counted twice
WHERE o.channel_key = 103 AND o.date_key = 20260904
UNION ALL
SELECT 'right: drill across (summarize each, then combine)', SUM(o.commission_amount_thb)
FROM (SELECT order_number, SUM(commission_amount_thb) AS commission_amount_thb FROM dw.fct_order
      WHERE channel_key = 103 AND date_key = 20260904 GROUP BY 1) o
JOIN (SELECT order_number, SUM(line_net_thb) AS items FROM dw.fct_order_line GROUP BY 1) l USING (order_number)
''')

,how,grab_commission_friday
0,right: order grain only,257.1
1,"wrong: two facts joined, then summed",403.5
2,"right: drill across (summarize each, then comb...",257.1


**The fix is in the design, not in the analyst's memory.** Order-level numbers live at order grain, line-level
numbers at line grain, and two facts are compared by **drilling across**: summarize each one to a common level
(a conformed dimension, or the degenerate `order_number`), then combine. When a number *must* be at line grain,
it gets **allocated down** instead of copied, which is what `FCT_ORDER_LINE_PROMOTION` does with a basket discount.

Every fact in the README starts with its grain sentence: *"one row per tender per order"*,
*"one row per status transition per order"*. If you can't say the grain in one sentence, the table isn't designed yet.

**วิธีแก้อยู่ในการออกแบบ ไม่ใช่ในความจำของนักวิเคราะห์** ตัวเลขระดับออร์เดอร์อยู่ที่ grain ออร์เดอร์ ตัวเลขระดับบรรทัดอยู่ที่ grain บรรทัด
และการเทียบ fact สองตัวทำด้วยการ **drill across**: สรุปแต่ละตัวให้อยู่ระดับเดียวกันก่อน (conformed dimension หรือ `order_number`) แล้วค่อยรวม
ถ้าตัวเลขไหน *ต้อง* อยู่ระดับบรรทัด ให้ **กระจายลงไป (allocate)** แทนการคัดลอก ซึ่งก็คือสิ่งที่ `FCT_ORDER_LINE_PROMOTION` ทำกับส่วนลดระดับตะกร้า
fact ทุกตัวใน README เริ่มจากประโยคบอก grain ถ้ายังพูด grain ไม่ได้ในประโยคเดียว แปลว่าตารางนั้นยังออกแบบไม่เสร็จ

## 3.2 · Rule 2: snapshot the numbers that can change later

`FCT_ORDER_LINE.unit_price_thb` copies the price **onto the fact** even though `DIM_MENU_ITEM` also has a
price. Chapter 5 asks this exact question (slide 21: is a unit price a fact or a dimension?). Here's why ours is both (README decision 03):

In [11]:
q('''
SELECT 'fact: what was actually paid'                      AS method, SUM(l.line_net_thb) AS pep_l_revenue_friday
FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key)
WHERE mi.menu_item_nk = 'PEP-L' AND l.date_key = 20260904
UNION ALL
SELECT 'qty x the CURRENT menu price (no snapshot, no SCD2)', SUM(l.quantity * cur.base_price_thb)
FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key)
JOIN dw.dim_menu_item cur ON cur.menu_item_nk = mi.menu_item_nk AND cur.current_flag = 'Current' AND cur.row_type = 'Menu item'
WHERE mi.menu_item_nk = 'PEP-L' AND l.date_key = 20260904
UNION ALL
SELECT 'qty x the list price valid that day (SCD2)',     SUM(l.quantity * mi.base_price_thb)
FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key)
WHERE mi.menu_item_nk = 'PEP-L' AND l.date_key = 20260904
''')

,method,pep_l_revenue_friday
0,fact: what was actually paid,1037.0
1,"qty x the CURRENT menu price (no snapshot, no ...",1047.0
2,qty x the list price valid that day (SCD2),987.0


Three different numbers, and each one answers a different question:

- **1,037**: what the customers actually paid. Only the snapshot on the fact knows this, because the Grab uplift (379) is in no dimension.
- **1,047**: what you'd get if the price were only in the dimension and you used today's price. It **rewrites Friday** with Saturday's price rise.
- **987**: Friday's list price × units. SCD2 answers *"what was true then"*, and comparing it to 1,037 shows how much the Grab uplift earned.

> **Snapshot answers *what happened*. SCD2 answers *what was true*.** Our model uses the same rule for
> `FCT_SHIFT.ot_multiplier_applied`, `FCT_ORDER_LINE_MODIFIER.price_delta_thb` and
> `FCT_PURCHASE_ORDER_LINE.unit_cost_thb`. Any number that is *policy*, and so can change, is copied onto the fact at the moment it happened.

ตัวเลขสามค่า ตอบสามคำถาม: **1,037** คือเงินที่ลูกค้าจ่ายจริง มีแค่ snapshot ใน fact ที่รู้ เพราะราคาที่ Grab บวกเพิ่ม (379) ไม่ได้อยู่ใน dimension ไหนเลย,
**1,047** คือค่าที่ได้ถ้าเก็บราคาไว้แค่ใน dimension แล้วใช้ราคาปัจจุบัน มัน **เขียนวันศุกร์ใหม่** ด้วยราคาที่ขึ้นวันเสาร์,
**987** คือราคาตั้งวันศุกร์ × จำนวน SCD2 ตอบว่า *"ตอนนั้นอะไรจริง"* และเทียบกับ 1,037 ก็จะเห็นว่าส่วนที่ Grab บวกเพิ่มทำเงินได้เท่าไร
**snapshot ตอบ *เกิดอะไรขึ้น* ส่วน SCD2 ตอบ *ตอนนั้นอะไรจริง*** ตัวเลขไหนเป็น *นโยบาย* ที่เปลี่ยนได้ ให้คัดลอกลง fact ณ ตอนที่มันเกิดขึ้น

## 3.3 · Rule 3: a list inside a column becomes its own fact table

The POS stored `"EXTRA_CHEESE+40;NO_CHILI+0"` as text. Our README turned that into
`FCT_ORDER_LINE_MODIFIER` (grain: one modifier on one order line), and the teacher's feedback was right
that the real damage of the text column is **inventory**, not tidiness. Let's build the stock facts and see it.

The movement fact follows the same conventions as everything else: dimension keys for analysis (menu item,
modifier, supplier, each **0 = Not applicable** when it does not apply), a `DIM_MOVEMENT_INFO` junk dimension for
the movement type, and a typed **degenerate** source document reference for lineage.

POS เก็บ `"EXTRA_CHEESE+40;NO_CHILI+0"` เป็น text แต่ README แปลงเป็น `FCT_ORDER_LINE_MODIFIER` (grain: modifier หนึ่งตัวบนบรรทัดออร์เดอร์หนึ่งบรรทัด)
และ feedback ของอาจารย์ก็ถูก: ความเสียหายจริงของคอลัมน์ text อยู่ที่ **สต็อก** ไม่ใช่แค่ความเรียบร้อย มาสร้าง fact ของสต็อกแล้วดูกัน
fact การเคลื่อนไหวสต็อกใช้ข้อตกลงเดียวกับตัวอื่น: key ของ dimension สำหรับวิเคราะห์ (เมนู modifier ซัพพลายเออร์ เป็น **0 = ไม่เกี่ยวข้อง** ถ้าไม่ใช้),
junk dimension `DIM_MOVEMENT_INFO` สำหรับประเภทการเคลื่อนไหว และรหัสเอกสารต้นทางแบบ **degenerate** สำหรับตามย้อนกลับ

In [12]:
# Junk dimension: movement type, direction, and what kind of document the movement came from
# junk dimension: ประเภทการเคลื่อนไหว ทิศทาง และชนิดของเอกสารต้นทาง
load("dw.dim_movement_info", pd.DataFrame([
    (101, "receipt",           "In",     "PO line"),
    (102, "theoretical_usage", "Out",    "Order line"),
    (103, "theoretical_usage", "Either", "Order line modifier"),   # no chili gives stock back  |  ไม่ใส่พริกคืนสต็อก
    (104, "count_adjustment",  "Either", "Stock count"),
], columns=["movement_info_key", "movement_type", "direction", "source_document_type"]))

# FCT_INVENTORY_MOVEMENT: grain = one stock movement of one ingredient. One signed measure, qty_delta.
# FCT_INVENTORY_MOVEMENT: grain = การเคลื่อนไหวสต็อกหนึ่งครั้งของวัตถุดิบหนึ่งตัว ตัวเลขเดียวที่มีเครื่องหมาย คือ qty_delta
con.execute('''
CREATE TABLE dw.fct_inventory_movement (
    ingredient_key         INTEGER,
    date_key               INTEGER,
    movement_info_key      INTEGER,
    menu_item_key          INTEGER,     -- theoretical usage, else 0  |  การใช้ตามทฤษฎี ไม่อย่างนั้นเป็น 0
    modifier_key           INTEGER,     -- modifier usage, else 0     |  การใช้ของ modifier ไม่อย่างนั้นเป็น 0
    supplier_key           INTEGER,     -- receipt, else 0            |  การรับของ ไม่อย่างนั้นเป็น 0
    source_document_number VARCHAR,     -- degenerate lineage: order, PO or count  |  รหัสเอกสารต้นทาง
    source_line_number     INTEGER,
    qty_delta              DECIMAL(10, 2)   -- signed: + in, - out  |  มีเครื่องหมาย: + เข้า, - ออก
)''')

# Opening balance from Thursday's count, then Friday's 2 kg delivery  |  ยอดยกมาจากการนับวันพฤหัส แล้วรับของ 2 กก. วันศุกร์
con.execute("INSERT INTO dw.fct_inventory_movement VALUES (101, 20260903, 104, 0, 0, 0,   'CNT-20260903', 0, 5000)")
con.execute("INSERT INTO dw.fct_inventory_movement VALUES (101, 20260904, 101, 0, 0, 101, 'PO-0001',      1, 2000)")

# Theoretical usage, explosion 1: order line x BRG_RECIPE  |  การใช้ตามทฤษฎี ชั้นที่ 1: บรรทัดออร์เดอร์ x สูตร
con.execute('''
INSERT INTO dw.fct_inventory_movement
SELECT r.ingredient_key, l.date_key, 102, l.menu_item_key, 0, 0, l.order_number, l.line_number, -l.quantity * r.qty_per_unit
FROM dw.fct_order_line l
JOIN dw.dim_order_info oi USING (order_info_key)
JOIN dw.dim_menu_item mi USING (menu_item_key) JOIN dw.brg_recipe r USING (menu_item_nk)
WHERE oi.cancellation_flag = 'Not cancelled'
''')
# Theoretical usage, explosion 2: modifier x BRG_MODIFIER_RECIPE (the one the text column could never do)
# การใช้ตามทฤษฎี ชั้นที่ 2: modifier x สูตรของ modifier (สิ่งที่คอลัมน์ text ทำไม่ได้เลย)
con.execute('''
INSERT INTO dw.fct_inventory_movement
SELECT mr.ingredient_key, m.date_key, 103, m.menu_item_key, m.modifier_key, 0, m.order_number, m.line_number,
       -l.quantity * m.quantity * mr.qty_delta_per_unit
FROM dw.fct_order_line_modifier m
JOIN dw.fct_order_line l USING (order_number, line_number)    -- ETL lineage lookup for the line quantity
JOIN dw.brg_modifier_recipe mr USING (modifier_key)
''')

# FCT_INVENTORY_COUNT: grain = one ingredient counted at one daily close. Each count books the gap as an adjustment.
# FCT_INVENTORY_COUNT: grain = วัตถุดิบหนึ่งตัวที่นับตอนปิดร้านหนึ่งวัน ทุกการนับจะลงส่วนต่างเป็น count_adjustment
con.execute('''
CREATE TABLE dw.fct_inventory_count (ingredient_key INTEGER, date_key INTEGER, count_session_id VARCHAR,
    qty_counted DECIMAL(10, 2), qty_expected DECIMAL(10, 2), variance_qty DECIMAL(10, 2))''')
con.execute("INSERT INTO dw.fct_inventory_count VALUES (101, 20260903, 'CNT-20260903', 5000, 0, 0)")
for date_key, counted in [(20260904, 6110), (20260905, 5205)]:
    expected = con.execute(f"SELECT SUM(qty_delta) FROM dw.fct_inventory_movement WHERE ingredient_key = 101 AND date_key <= {date_key}").fetchone()[0]
    con.execute(f"INSERT INTO dw.fct_inventory_count VALUES (101, {date_key}, 'CNT-{date_key}', {counted}, {expected}, {counted} - {expected})")
    con.execute(f"INSERT INTO dw.fct_inventory_movement VALUES (101, {date_key}, 104, 0, 0, 0, 'CNT-{date_key}', 0, {counted} - {expected})")

q("SELECT * FROM dw.fct_inventory_count")

,ingredient_key,date_key,count_session_id,qty_counted,qty_expected,variance_qty
0,101,20260903,CNT-20260903,5000.0,0.0,0.0
1,101,20260904,CNT-20260904,6110.0,6130.0,-20.0
2,101,20260905,CNT-20260905,5205.0,5220.0,-15.0


In [13]:
# What the variance would have looked like if modifiers were still a text column (usage never counted)
# variance จะออกมาเป็นยังไง ถ้า modifier ยังเป็นคอลัมน์ text อยู่ (การใช้ของ modifier ไม่ถูกนับเลย)
q('''
WITH usage AS (
    SELECT m.date_key,
           SUM(m.qty_delta) FILTER (WHERE mi.source_document_type = 'Order line')          AS recipe_usage,
           SUM(m.qty_delta) FILTER (WHERE mi.source_document_type = 'Order line modifier') AS modifier_usage
    FROM dw.fct_inventory_movement m JOIN dw.dim_movement_info mi USING (movement_info_key)
    WHERE m.ingredient_key = 101 GROUP BY 1
)
SELECT c.date_key, u.recipe_usage, u.modifier_usage, c.qty_counted,
       c.variance_qty                                    AS variance_with_modifier_fact,
       c.variance_qty + u.modifier_usage                 AS variance_if_modifiers_were_text
FROM dw.fct_inventory_count c JOIN usage u USING (date_key)
ORDER BY 1
''')

,date_key,recipe_usage,modifier_usage,qty_counted,variance_with_modifier_fact,variance_if_modifiers_were_text
0,20260903,NaN,NaN,5000.0,0.0,NaN
1,20260904,-790.0,-80.0,6110.0,-20.0,-100.0
2,20260905,-850.0,-40.0,5205.0,-15.0,-55.0


With the modifier fact, Friday's unexplained mozzarella loss is **−20 g** (a bit of waste). Without it,
it's **−100 g**: five times worse, and the manager would go looking for a thief when the real answer is
"customers like extra cheese". **A list stuffed into a column breaks the join that a downstream number depends on.**
The same argument split `DIM_SUPPLIER.delivery_days` into `BRG_SUPPLIER_DELIVERY_DAY` (decision 09).

ถ้ามี fact ของ modifier มอซซาเรลลาที่หายไปแบบอธิบายไม่ได้ในวันศุกร์คือ **−20 กรัม** (ของเสียนิดหน่อย) ถ้าไม่มี จะเป็น **−100 กรัม** แย่กว่าห้าเท่า
และผู้จัดการจะไปตามหาคนขโมย ทั้งที่คำตอบจริงคือ "ลูกค้าชอบสั่งชีสเพิ่ม" **list ที่ยัดไว้ในคอลัมน์เดียวทำให้ join ที่ตัวเลขปลายทางต้องใช้พังไป**

## 3.4 · Rule 4: know which measures you are allowed to add up

| Kind | Can you `SUM` it… | Pizza shop examples |
|---|---|---|
| **Additive** | across every dimension | `quantity`, `line_net_thb`, `qty_delta`, `labour_cost_thb` |
| **Semi-additive** | across some dimensions, **never across time** | `qty_counted`, stock on hand: a *level*, not a flow |
| **Nonadditive** | never; break it down into additive parts (slide 33) | `unit_price_thb`, `commission_rate_pct`, any % or ratio |

This is why `FCT_INVENTORY_MOVEMENT` stores one signed `qty_delta` (additive: stock on hand = `SUM(qty_delta)`,
one expression with nothing to forget), while `FCT_INVENTORY_COUNT` stores a level that you must never add across days.

**Additive** sum ได้ทุก dimension **Semi-additive** sum ได้บาง dimension แต่ **ห้าม sum ข้ามเวลา** (สต็อกคงเหลือเป็น *ระดับ* ไม่ใช่ *การไหล*)
**Nonadditive** sum ไม่ได้เลย ต้องแยกเป็นส่วนที่บวกได้ (สไลด์ 33) นี่คือเหตุผลที่ `FCT_INVENTORY_MOVEMENT` เก็บ `qty_delta` ตัวเดียวแบบมีเครื่องหมาย
ส่วน `FCT_INVENTORY_COUNT` เก็บ *ระดับ* ที่ห้ามบวกข้ามวัน

In [14]:
display(q('''
SELECT 'WRONG: SUM of counts over Thu-Sat'      AS how, SUM(qty_counted) AS mozzarella_g FROM dw.fct_inventory_count
UNION ALL
SELECT 'right: the last count in the period',  arg_max(qty_counted, date_key) FROM dw.fct_inventory_count
UNION ALL
SELECT 'right: SUM of movements (additive)',   SUM(qty_delta) FROM dw.fct_inventory_movement WHERE ingredient_key = 101
'''))

display(q('''
SELECT 'WRONG: AVG(unit_price)'                AS how, round(AVG(l.unit_price_thb), 2) AS avg_pep_l_price
FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key) WHERE mi.menu_item_nk = 'PEP-L'
UNION ALL
SELECT 'right: SUM(revenue) / SUM(units)',     round(SUM(l.line_net_thb) / SUM(l.quantity), 2)
FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key) WHERE mi.menu_item_nk = 'PEP-L'
'''))

,how,mozzarella_g
0,WRONG: SUM of counts over Thu-Sat,16315.0
1,right: the last count in the period,5205.0
2,right: SUM of movements (additive),5205.0


,how,avg_pep_l_price
0,WRONG: AVG(unit_price),357.00
1,right: SUM(revenue) / SUM(units),361.86


"We have 16 kg of cheese" is a sentence that sends a manager to an empty fridge. And `AVG(unit_price)`
treats a line of 2 pizzas the same as a line of 1. **Store the additive parts (revenue and units), and
compute ratios at query time.** That's why `FCT_SHIFT` stores `regular_hours`, `ot_hours` and the costs
separately instead of one blended rate (decision 08).

"เรามีชีส 16 กิโล" คือประโยคที่ส่งผู้จัดการไปเจอตู้เย็นว่างเปล่า ส่วน `AVG(unit_price)` นับบรรทัดที่สั่ง 2 ถาดเท่ากับบรรทัดที่สั่ง 1 ถาด
**เก็บส่วนที่บวกได้ (ยอดเงินและจำนวน) แล้วค่อยคำนวณอัตราส่วนตอน query** นี่คือเหตุผลที่ `FCT_SHIFT` เก็บ `regular_hours`, `ot_hours` และค่าแรงแยกกัน (decision 08)

## 3.5 · Rule 5: no NULL keys; facts point at dimensions, never at other facts

Grab orders have no customer. The tempting design stores `customer_key = NULL`. Chapter 5 (slides 69–71) says
not to: give the dimension a **special-case row** and point at that. Here is what the NULL would have done to a
perfectly ordinary report, revenue by membership:

ออร์เดอร์ Grab ไม่มีลูกค้า การออกแบบที่ล่อใจคือเก็บ `customer_key = NULL` บทที่ 5 (สไลด์ 69–71) บอกว่าไม่ควร ให้เพิ่ม **special-case row** ใน dimension แล้วชี้ไปที่แถวนั้น
นี่คือสิ่งที่ NULL จะทำกับรายงานธรรมดา ๆ อย่างรายได้ตามสถานะสมาชิก:

In [15]:
q('''
WITH with_null AS (       -- the design we did NOT choose: NULL instead of row 0  |  แบบที่เราไม่ได้เลือก: NULL แทนแถว 0
    SELECT CASE WHEN customer_key = 0 THEN NULL ELSE customer_key END AS customer_key, net_revenue_thb FROM dw.fct_order
)
SELECT 'with row 0 (Not applicable)' AS design, cu.membership, cu.row_type, SUM(o.net_revenue_thb) AS net_thb
FROM dw.fct_order o JOIN dw.dim_customer cu USING (customer_key) GROUP BY ALL
UNION ALL
SELECT 'with NULL keys', cu.membership, cu.row_type, SUM(n.net_revenue_thb)
FROM with_null n JOIN dw.dim_customer cu USING (customer_key) GROUP BY ALL
UNION ALL
SELECT 'total in FCT_ORDER', NULL, NULL, SUM(net_revenue_thb) FROM dw.fct_order
ORDER BY 1, 2
''')

,design,membership,row_type,net_thb
0,total in FCT_ORDER,NaN,NaN,3782.0
1,with NULL keys,Member,Customer,1455.0
2,with row 0 (Not applicable),Member,Customer,1455.0
3,with row 0 (Not applicable),n/a,Not applicable,2327.0


With NULL keys, **2,327 THB of the 3,782 THB vanishes** from the report, with no error: every order with no identified
customer (all three Grab orders and the two walk-in QR orders) is dropped by the inner join, and the report sums to 1,455.
With row 0 those orders show up honestly as *Not applicable*, and the report still adds up to the total.
The same row 1, *Unknown*, catches late-arriving data (Part 2.5), and row 2, *Invalid*, catches bad codes.

The second half of the rule: **facts point at dimensions, never at other facts.** `FCT_ORDER_LINE` carries its own
date, channel and customer keys instead of an `order_key` back to `FCT_ORDER`, and `FCT_INVENTORY_MOVEMENT` says
where stock went through `menu_item_key`, `modifier_key` and `supplier_key`, not through keys into the order-line or PO
facts. Its lineage is a typed **degenerate** reference (`source_document_number`, with its type in `DIM_MOVEMENT_INFO`),
checked by the ETL (chapter 5, slide 27: "referential integrity can be enforced by the ETL process"). No analysis
query joins on it, so the wrong-table join that made the original polymorphic `source_ref_sk` dangerous cannot happen.

ถ้าใช้ NULL **รายได้ 2,327 จาก 3,782 บาทจะหายไปจากรายงาน** โดยไม่มี error เลย ออร์เดอร์ที่ไม่มีลูกค้าระบุตัว (ออร์เดอร์ Grab ทั้งสามใบ และออร์เดอร์ QR ของลูกค้า walk-in สองใบ) ถูก inner join ตัดทิ้ง รายงานรวมได้แค่ 1,455
ถ้าใช้แถว 0 ออร์เดอร์เหล่านั้นจะแสดงตรง ๆ ว่า *ไม่เกี่ยวข้อง* และรายงานยังรวมได้เท่ายอดรวม
แถว 1 *ไม่ทราบ* ใช้รับข้อมูลที่มาก่อนบริบท (Part 2.5) และแถว 2 *ไม่ถูกต้อง* ใช้รับรหัสที่ผิด
ครึ่งหลังของกฎ: **fact ชี้ไปที่ dimension ไม่ชี้ไปที่ fact ตัวอื่น** `FCT_ORDER_LINE` มี key วันที่ ช่องทาง ลูกค้าเอง ไม่มี `order_key` ย้อนไปที่ `FCT_ORDER`
และ `FCT_INVENTORY_MOVEMENT` บอกว่าสต็อกไปไหนผ่าน `menu_item_key`, `modifier_key`, `supplier_key` ส่วนการตามย้อนกลับใช้รหัสเอกสารแบบ **degenerate**
(`source_document_number` โดยชนิดอยู่ใน `DIM_MOVEMENT_INFO`) ซึ่ง ETL เป็นคนตรวจ (สไลด์ 27) ไม่มี query วิเคราะห์ไหน join ด้วยคอลัมน์นี้ การ join ผิดตารางแบบ `source_ref_sk` เดิมจึงเกิดขึ้นไม่ได้

## 3.6 · The four types of fact table, and where ours sit

| Type | One row is… | Rows change? | Ours |
|---|---|---|---|
| **Transaction** | one event, at the moment it happened | never, append only | `FCT_ORDER_LINE`, `FCT_PAYMENT`, `FCT_ORDER_EVENT`, `FCT_INVENTORY_MOVEMENT`, `FCT_PURCHASE_ORDER_LINE`, `FCT_SHIFT` |
| **Periodic snapshot** | the state of something at a regular interval | never, one new row per period | `FCT_INVENTORY_COUNT` (daily close) |
| **Accumulating snapshot** | one process instance with a column per milestone | **yes**, updated as milestones happen | `mart_delivery_sla` (built from `FCT_ORDER_EVENT`) |
| **Factless** | that something happened or was possible, with no number | rarely | `BRG_CHANNEL_MENU_ITEM` works as a coverage table |

**Transaction** = หนึ่งเหตุการณ์ ณ เวลาที่เกิด เพิ่มอย่างเดียว **Periodic snapshot** = สถานะของบางอย่างทุกช่วงเวลาที่กำหนด (เช่น ทุกปิดร้าน)
**Accumulating snapshot** = หนึ่งกระบวนการ หนึ่งคอลัมน์ต่อหนึ่งขั้น และ **ถูกอัปเดต** เมื่อถึงขั้นถัดไป **Factless** = บันทึกว่าบางอย่างเกิดขึ้นหรือเป็นไปได้ โดยไม่มีตัวเลข

### Periodic snapshot from transactions

A periodic snapshot can always be **derived** from a transaction fact by running a sum through time. That's
why our movements are the source of truth and the count is the check:

periodic snapshot สร้างจาก transaction fact ได้เสมอด้วยการบวกสะสมตามเวลา นั่นคือเหตุผลที่ movement เป็นแหล่งความจริง และการนับเป็นตัวตรวจ:

In [16]:
q('''
SELECT d.full_date, i.ingredient_nk,
       SUM(m.qty_delta)                                                    AS net_movement_that_day,
       SUM(SUM(m.qty_delta)) OVER (PARTITION BY i.ingredient_nk ORDER BY d.full_date) AS on_hand_at_close
FROM dw.fct_inventory_movement m
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_ingredient i USING (ingredient_key)
GROUP BY d.full_date, i.ingredient_nk ORDER BY i.ingredient_nk, d.full_date
''')

,full_date,ingredient_nk,net_movement_that_day,on_hand_at_close
0,2026-09-04,chili,-10.0,-10.0
1,2026-09-05,chili,-25.0,-35.0
2,2026-09-03,mozzarella,5000.0,5000.0
3,2026-09-04,mozzarella,1110.0,6110.0
4,2026-09-05,mozzarella,-905.0,5205.0
5,2026-09-04,pepperoni,-180.0,-180.0
6,2026-09-05,pepperoni,-280.0,-460.0


Mozzarella's `on_hand_at_close` matches the counts exactly (5000, 6110, 5205), because each count booked its
gap as a `count_adjustment`. Pepperoni and chili have no opening count in this demo, so they run negative.
That's a data quality finding in its own right: usage with no opening balance.

`on_hand_at_close` ของมอซซาเรลลาตรงกับการนับเป๊ะ (5000, 6110, 5205) เพราะทุกการนับลงส่วนต่างเป็น `count_adjustment`
ส่วนเปปเปอโรนีกับพริกไม่มีการนับยอดยกมาในตัวอย่างนี้ จึงติดลบ ซึ่งก็คือ data quality finding อย่างหนึ่ง: มีการใช้ แต่ไม่มียอดตั้งต้น

### Accumulating snapshot from an event log

Our README stores delivery timing as an **event log** (`FCT_ORDER_EVENT`, one row per status change; decision 05)
and builds the **accumulating snapshot** as an aggregate table. Why not store the snapshot directly? A new milestone
(say Grab adds `driver_arrived`) is a new *row* in `DIM_EVENT_TYPE`, not a schema change. And the log never needs updating.

README เก็บเวลาการจัดส่งเป็น **event log** (`FCT_ORDER_EVENT` หนึ่งแถวต่อการเปลี่ยนสถานะหนึ่งครั้ง; decision 05) แล้วสร้าง **accumulating snapshot** เป็น aggregate table
ทำไมไม่เก็บ snapshot ตรง ๆ? เพราะถ้ามีขั้นใหม่ (เช่น Grab เพิ่ม `driver_arrived`) ก็เป็นแค่ *แถวใหม่* ใน `DIM_EVENT_TYPE` ไม่ต้องแก้ schema และ log ไม่ต้องถูกอัปเดตเลย

In [17]:
load("dw.dim_event_type", pd.DataFrame([
    (101, "placed", 1, "Kitchen"), (102, "accepted", 2, "Kitchen"), (103, "prep_start", 3, "Kitchen"),
    (104, "prep_complete", 4, "Kitchen"), (105, "courier_pickup", 5, "Delivery"), (106, "delivered", 6, "Delivery"),
], columns=["event_type_key", "event_type", "event_sequence", "event_stage"]))

# FCT_ORDER_EVENT: grain = one status transition of one order. Grab only reports accepted / pickup / delivered.
# FCT_ORDER_EVENT: grain = การเปลี่ยนสถานะหนึ่งครั้งของออร์เดอร์หนึ่งใบ Grab ส่งมาแค่ accepted / pickup / delivered
events = pd.DataFrame([
    ("P-1001", "placed", "2026-09-04 12:10"), ("P-1001", "accepted", "2026-09-04 12:11"),
    ("P-1001", "prep_start", "2026-09-04 12:12"), ("P-1001", "prep_complete", "2026-09-04 12:24"),
    ("GF-201", "placed", "2026-09-04 18:30"), ("GF-201", "accepted", "2026-09-04 18:31"),
    ("GF-201", "courier_pickup", "2026-09-04 18:52"), ("GF-201", "delivered", "2026-09-04 19:10"),
    ("GF-202", "placed", "2026-09-05 01:10"), ("GF-202", "accepted", "2026-09-05 01:12"),
    ("GF-202", "courier_pickup", "2026-09-05 01:35"), ("GF-202", "delivered", "2026-09-05 01:58"),
    ("GF-203", "placed", "2026-09-05 13:00"), ("GF-203", "accepted", "2026-09-05 13:02"),
    ("GF-203", "courier_pickup", "2026-09-05 13:20"),                      # still on its way  |  ยังส่งไม่ถึง
], columns=["order_number", "event_type", "event_ts"])
events["event_ts"] = pd.to_datetime(events["event_ts"])
con.register("_events", events)
con.execute('''
CREATE TABLE dw.fct_order_event AS
SELECT k.date_key, k.channel_key, et.event_type_key,
       0 AS staff_key,                                   -- 0 = Not applicable: no staff recorded in this demo
       e.order_number, e.event_ts
FROM _events e
JOIN stg.order_keys k USING (order_number)
JOIN dw.dim_event_type et USING (event_type)
''')

# The accumulating snapshot: one row per order, one column per milestone  |  หนึ่งแถวต่อออร์เดอร์ หนึ่งคอลัมน์ต่อหนึ่งขั้น
q('''
SELECT e.order_number,
       MAX(event_ts) FILTER (WHERE et.event_type = 'placed')          AS placed,
       MAX(event_ts) FILTER (WHERE et.event_type = 'prep_complete')   AS prep_complete,
       MAX(event_ts) FILTER (WHERE et.event_type = 'courier_pickup')  AS courier_pickup,
       MAX(event_ts) FILTER (WHERE et.event_type = 'delivered')       AS delivered,
       date_diff('minute', placed, prep_complete)                     AS prep_min,
       date_diff('minute', placed, courier_pickup)                    AS to_pickup_min,
       date_diff('minute', placed, delivered)                         AS lead_time_min
FROM dw.fct_order_event e JOIN dw.dim_event_type et USING (event_type_key)
GROUP BY e.order_number ORDER BY placed
''')

,order_number,placed,prep_complete,courier_pickup,delivered,prep_min,to_pickup_min,lead_time_min
0,P-1001,2026-09-04 12:10:00,2026-09-04 12:24:00,NaT,NaT,14,<NA>,<NA>
1,GF-201,2026-09-04 18:30:00,NaT,2026-09-04 18:52:00,2026-09-04 19:10:00,<NA>,22,40
2,GF-202,2026-09-05 01:10:00,NaT,2026-09-05 01:35:00,2026-09-05 01:58:00,<NA>,25,48
3,GF-203,2026-09-05 13:00:00,NaT,2026-09-05 13:20:00,NaT,<NA>,20,<NA>


Two things to notice. **GF-203 has no `delivered` yet**: in a stored accumulating snapshot, that row
would be *updated* later, the only kind of fact row that changes. **Grab has no prep times** and QR has
no courier times. The NULLs here are *measurements* that never happened, not missing keys, and they are honest about
what each source tells us (README decision 05).

สองอย่างที่ควรสังเกต: **GF-203 ยังไม่มี `delivered`** ถ้าเก็บเป็น accumulating snapshot แถวนี้จะถูก *อัปเดต* ทีหลัง
และ **Grab ไม่มีเวลาทำอาหาร** ส่วน QR ไม่มีเวลาไรเดอร์ ค่า NULL ตรงนี้คือ *การวัด* ที่ไม่ได้เกิดขึ้น ไม่ใช่ key ที่หายไป

### Factless: what *didn't* happen

A coverage table records what was *possible*. `BRG_CHANNEL_MENU_ITEM` says what each channel lists.
Subtract what sold, and you get what was listed but never ordered, which no sales fact could tell you:

ตาราง coverage บันทึกสิ่งที่ *เป็นไปได้* `BRG_CHANNEL_MENU_ITEM` บอกว่าแต่ละช่องทางลงขายอะไรบ้าง ลบด้วยสิ่งที่ขายได้
ก็จะได้เมนูที่ลงขายแต่ไม่มีใครสั่ง ซึ่ง fact การขายตัวไหนก็บอกไม่ได้:

In [18]:
q('''
SELECT c.channel_code, b.menu_item_nk, b.platform_item_name, b.channel_price_thb
FROM dw.brg_channel_menu_item b JOIN dw.dim_channel c USING (channel_key)
WHERE b.listing_status = 'Listed'
  AND NOT EXISTS (
      SELECT 1 FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key)
      WHERE l.channel_key = b.channel_key AND mi.menu_item_nk = b.menu_item_nk)
ORDER BY 1, 2
''')

,channel_code,menu_item_nk,platform_item_name,channel_price_thb
0,GRAB,PEP-M,Pepperoni Pizza (M),289.0
1,QR_DINEIN,GARLIC,GARLIC,NaN
2,QR_DINEIN,HAW-L,HAW-L,NaN
3,QR_DINEIN,PEP-M,PEP-M,NaN


Pepperoni M on Grab got no orders at all. The three QR rows are subtler: those items *did* sell, but only by phone.
That's the kind of finding ("Hawaiian never sells at the table") that only a coverage table can produce.

Pepperoni M บน Grab ไม่มีใครสั่งเลย ส่วนสามแถวของ QR ละเอียดกว่านั้น: เมนูพวกนี้ขายได้ แต่ขายทางโทรศัพท์อย่างเดียว
ข้อค้นพบแบบนี้ ("ฮาวายเอี้ยนไม่เคยขายได้ที่โต๊ะ") ได้มาจากตาราง coverage เท่านั้น

## 3.7 · The catalogue: every fact table in our model, and why

This is the "how and why" for each fact table in the README, using the four steps from 2.1.
Use it as the cheat sheet when someone in the review asks *"why is this a separate table?"*

นี่คือ "ทำยังไงและทำไม" ของ fact ทุกตัวใน README ตาม 4 ขั้นตอนใน 2.1 ใช้เป็นโพยตอนมีคนถามในการรีวิวว่า *"ทำไมต้องแยกเป็นอีกตาราง?"*

In [19]:
catalogue = pd.DataFrame([
    ("FCT_ORDER", "transaction", "one order, any channel",
     "date, channel, customer, order info", "gross, discount, delivery_fee, commission, net_revenue (all additive)",
     "Order-level money must not repeat per line (3.1). Net revenue = what the shop banks, differs ~30% by channel."),
    ("FCT_ORDER_LINE", "transaction", "one menu item on one order",
     "date, channel, customer, menu item, order info", "quantity, line_net (additive); unit_price (nonadditive, snapshot)",
     "Forecast grain is item x day x channel, so sales must be item-level. Carries its own keys; price snapshotted (3.2)."),
    ("FCT_ORDER_LINE_MODIFIER", "transaction", "one modifier on one order line",
     "date, channel, menu item, modifier", "quantity (additive); price_delta (snapshot)",
     "Was modifiers_json. Without it, extra cheese never counts as usage and inflates variance 5x (3.3)."),
    ("FCT_PAYMENT", "transaction", "one tender on one order",
     "paid date & settlement date (roles), channel, customer, payment info", "amount_thb (additive)",
     "Split payments are normal (half PromptPay, half cash). Settlement date is 0 (Not applicable) until paid out."),
    ("FCT_ORDER_EVENT", "transaction (event log)", "one status change of one order",
     "date, channel, event type, staff", "event_ts; duration_from_prev_sec (additive within an order)",
     "Any duration is a difference of two rows; a new milestone is a new DIM_EVENT_TYPE row (3.6)."),
    ("FCT_ORDER_PROMOTION", "transaction", "one promotion on one order",
     "date, channel, promotion", "discount_amount (additive)",
     "Basket-level discount as the platform reports it. Reconciles with the line-level table by drilling across."),
    ("FCT_ORDER_LINE_PROMOTION", "transaction", "one promotion on one order line",
     "date, channel, customer, menu item, promotion, promo application", "discount_amount (additive, allocated)",
     "Pushes discount down to SKU so item margin is real. Allocated, not copied (3.1)."),
    ("FCT_PURCHASE_ORDER", "transaction", "one PO to one supplier",
     "supplier, order date & received date (roles), PO info", "total_cost (additive)",
     "Two keys to the same DIM_DATE: dimension roles, accessed through aliases."),
    ("FCT_PURCHASE_ORDER_LINE", "transaction", "one ingredient on one PO",
     "supplier, ingredient, order & received date, PO info", "qty_ordered, qty_received (additive); unit_cost (snapshot)",
     "Cost at time of purchase, so food cost % doesn't drift when prices change."),
    ("FCT_INVENTORY_MOVEMENT", "transaction", "one stock movement of one ingredient",
     "ingredient, date, movement info, menu item, modifier, supplier", "qty_delta (signed, additive)",
     "Stock on hand = SUM(qty_delta). Dimension keys for analysis (0 = n/a), typed degenerate reference for lineage (3.5)."),
    ("FCT_INVENTORY_COUNT", "periodic snapshot", "one ingredient at one daily close",
     "ingredient, date, counted-by staff (role)", "qty_counted, qty_expected (semi-additive); variance (additive over items)",
     "Physical truth. Variance = counted - expected is the KPI the model exists for (3.4)."),
    ("FCT_SHIFT", "transaction", "one shift worked by one staff member",
     "staff, date, shift info", "regular_hours, ot_hours, costs (additive); ot_multiplier (snapshot)",
     "OT costs more per hour, so it's split out; the multiplier is policy, so it's snapshotted."),
    ("fct_forecast", "periodic snapshot", "item x day x channel x forecast run",
     "menu item, date, channel, run", "forecast_units (additive, but NOT across runs)",
     "Predicted vs actual becomes a table you can chart (3.8)."),
], columns=["fact table", "type", "grain (one row = ...)", "dimensions", "facts (measures)", "why it exists / why this shape"])

with pd.option_context("display.max_colwidth", 120):
    display(catalogue.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))

fact table,type,grain (one row = ...),dimensions,facts (measures),why it exists / why this shape
FCT_ORDER,transaction,"one order, any channel","date, channel, customer, order info","gross, discount, delivery_fee, commission, net_revenue (all additive)","Order-level money must not repeat per line (3.1). Net revenue = what the shop banks, differs ~30% by channel."
FCT_ORDER_LINE,transaction,one menu item on one order,"date, channel, customer, menu item, order info","quantity, line_net (additive); unit_price (nonadditive, snapshot)","Forecast grain is item x day x channel, so sales must be item-level. Carries its own keys; price snapshotted (3.2)."
FCT_ORDER_LINE_MODIFIER,transaction,one modifier on one order line,"date, channel, menu item, modifier",quantity (additive); price_delta (snapshot),"Was modifiers_json. Without it, extra cheese never counts as usage and inflates variance 5x (3.3)."
FCT_PAYMENT,transaction,one tender on one order,"paid date & settlement date (roles), channel, customer, payment info",amount_thb (additive),"Split payments are normal (half PromptPay, half cash). Settlement date is 0 (Not applicable) until paid out."
FCT_ORDER_EVENT,transaction (event log),one status change of one order,"date, channel, event type, staff",event_ts; duration_from_prev_sec (additive within an order),Any duration is a difference of two rows; a new milestone is a new DIM_EVENT_TYPE row (3.6).
FCT_ORDER_PROMOTION,transaction,one promotion on one order,"date, channel, promotion",discount_amount (additive),Basket-level discount as the platform reports it. Reconciles with the line-level table by drilling across.
FCT_ORDER_LINE_PROMOTION,transaction,one promotion on one order line,"date, channel, customer, menu item, promotion, promo application","discount_amount (additive, allocated)","Pushes discount down to SKU so item margin is real. Allocated, not copied (3.1)."
FCT_PURCHASE_ORDER,transaction,one PO to one supplier,"supplier, order date & received date (roles), PO info",total_cost (additive),"Two keys to the same DIM_DATE: dimension roles, accessed through aliases."
FCT_PURCHASE_ORDER_LINE,transaction,one ingredient on one PO,"supplier, ingredient, order & received date, PO info","qty_ordered, qty_received (additive); unit_cost (snapshot)","Cost at time of purchase, so food cost % doesn't drift when prices change."
FCT_INVENTORY_MOVEMENT,transaction,one stock movement of one ingredient,"ingredient, date, movement info, menu item, modifier, supplier","qty_delta (signed, additive)","Stock on hand = SUM(qty_delta). Dimension keys for analysis (0 = n/a), typed degenerate reference for lineage (3.5)."


## 3.8 · Worked example: designing a new fact table from scratch

The README mentions `fct_forecast` but never designs it. Let's do it with the four steps, so the
group can see the method working on something new. (Slide 55's conformance matrix has the same idea: a
`demand_forecast_facts` row that conforms on fewer levels than the sales facts.)

1. **Business process.** The forecast model *predicts* demand. A prediction is an event: it happened at a time, with a result.
2. **Grain.** *One forecast of one menu item, for one business day, on one channel, from one model run.*
   The **run** is part of the grain because we re-forecast every night, and the Monday-for-Saturday forecast and the Friday-for-Saturday forecast are different facts.
3. **Dimensions.** `DIM_MENU_ITEM`, `DIM_DATE` (the day being forecast), `DIM_CHANNEL`, and a run identifier with its own timestamp.
   All three dimensions are **conformed**, already built for sales. That's what lets us drill across from forecast to actual.
4. **Facts.** `forecast_units`. Additive across items, days and channels, but **not across runs**: summing two runs of the same Saturday double-counts.
   That makes it semi-additive, with the run playing the role that time plays for stock.

README พูดถึง `fct_forecast` แต่ไม่เคยออกแบบ มาออกแบบด้วย 4 ขั้นตอนกัน (conformance matrix ในสไลด์ 55 ก็มีแถว `demand_forecast_facts` แบบเดียวกัน):
1. **Business process** โมเดลพยากรณ์ *ทำนาย* ความต้องการ การทำนายหนึ่งครั้งก็คือเหตุการณ์หนึ่ง มีเวลาที่เกิดและมีผลลัพธ์
2. **Grain** *การพยากรณ์หนึ่งครั้ง ของเมนูหนึ่งรายการ สำหรับวันทำการหนึ่งวัน ในช่องทางหนึ่ง จากการรันโมเดลหนึ่งครั้ง* ต้องมี **run** อยู่ใน grain เพราะเราพยากรณ์ใหม่ทุกคืน
3. **Dimensions** `DIM_MENU_ITEM`, `DIM_DATE` (วันที่ถูกพยากรณ์), `DIM_CHANNEL` และรหัสการรัน ทั้งสามเป็น **conformed** ที่สร้างไว้แล้วสำหรับการขาย จึง drill across จาก forecast ไปหายอดจริงได้เลย
4. **Facts** `forecast_units` บวกข้ามเมนู วัน และช่องทางได้ แต่ **บวกข้าม run ไม่ได้** จึงเป็น semi-additive โดย run ทำหน้าที่เหมือนเวลาในกรณีของสต็อก

In [20]:
con.execute('''
CREATE TABLE dw.fct_forecast (
    forecast_run_id  VARCHAR,     -- degenerate: which nightly run  |  degenerate: การรันคืนไหน
    run_ts           TIMESTAMP,
    menu_item_nk     VARCHAR,     -- simplified; would be menu_item_key  |  ฉบับย่อ ของจริงจะเป็น menu_item_key
    date_key         INTEGER,     -- the day being forecast, not the day of the run  |  วันที่ถูกพยากรณ์ ไม่ใช่วันที่รัน
    channel_key      INTEGER,
    forecast_units   DECIMAL(6, 1)
)''')
con.execute('''
INSERT INTO dw.fct_forecast VALUES
  ('run-0901', '2026-09-01 03:00', 'PEP-L', 20260905, 101, 1.0),
  ('run-0901', '2026-09-01 03:00', 'PEP-L', 20260905, 103, 1.0),
  ('run-0904', '2026-09-04 03:00', 'PEP-L', 20260905, 101, 2.0),
  ('run-0904', '2026-09-04 03:00', 'PEP-L', 20260905, 103, 3.0)
''')

# Drill across: summarize actuals to the forecast's grain first, then combine  |  drill across: สรุปยอดจริงให้อยู่ grain เดียวกับ forecast ก่อน แล้วค่อยรวม
display(q('''
WITH actual AS (
    SELECT l.date_key, l.channel_key, mi.menu_item_nk, SUM(l.quantity) AS actual_units
    FROM dw.fct_order_line l JOIN dw.dim_menu_item mi USING (menu_item_key)
    GROUP BY ALL
)
SELECT f.forecast_run_id, c.channel_code, f.forecast_units, a.actual_units,
       f.forecast_units - a.actual_units AS error_units
FROM dw.fct_forecast f
JOIN dw.dim_channel c USING (channel_key)
LEFT JOIN actual a USING (date_key, channel_key, menu_item_nk)
ORDER BY f.forecast_run_id, c.channel_code
'''))

display(q('''
SELECT 'WRONG: sum over all runs'  AS how, SUM(forecast_units) AS saturday_pep_l_forecast FROM dw.fct_forecast
UNION ALL
SELECT 'right: latest run only', SUM(forecast_units) FROM dw.fct_forecast
WHERE forecast_run_id = (SELECT arg_max(forecast_run_id, run_ts) FROM dw.fct_forecast)
'''))

,forecast_run_id,channel_code,forecast_units,actual_units,error_units
0,run-0901,GRAB,1.0,2.0,-1.0
1,run-0901,QR_DINEIN,1.0,2.0,-1.0
2,run-0904,GRAB,3.0,2.0,1.0
3,run-0904,QR_DINEIN,2.0,2.0,0.0


,how,saturday_pep_l_forecast
0,WRONG: sum over all runs,7.0
1,right: latest run only,5.0


The later run (`run-0904`) was closer: it predicted 2 + 3 and actual was 2 + 2. Because forecasts are
facts at a declared grain on conformed dimensions, *"is the model getting better?"* is one `GROUP BY forecast_run_id` away.

การรันที่ใกล้วันกว่า (`run-0904`) แม่นกว่า ทำนาย 2 + 3 ส่วนของจริงคือ 2 + 2 เพราะ forecast เป็น fact ที่มี grain ชัดเจนบน conformed dimension
คำถาม *"โมเดลแม่นขึ้นไหม?"* จึงห่างแค่ `GROUP BY forecast_run_id` เดียว

> **Try it.** Design `fct_waste` yourselves: the kitchen logs dough or cheese thrown away at close.
> Write the four steps before any SQL. Hint: is it a new fact table, or a new row in `DIM_MOVEMENT_INFO`?
>
> **ลองดู** ออกแบบ `fct_waste` เอง: ครัวจดแป้งหรือชีสที่ทิ้งตอนปิดร้าน เขียน 4 ขั้นตอนให้เสร็จก่อนเขียน SQL
> คำใบ้: มันควรเป็น fact ตัวใหม่ หรือเป็นแถวใหม่ใน `DIM_MOVEMENT_INFO`?

---
## Summary

**Why a warehouse (ทำไปเพื่ออะไร):** integrate many sources into one shape, define business rules once,
keep history, shape data for questions, and keep heavy analysis off the POS.

**How (ทำออกมายังไง):** a Kimball dimensional warehouse; business process → grain → dimensions → facts; a
conformance matrix of conformed dimensions and their hierarchies; `_key` surrogate keys, SCD2 and special-case rows;
load facts by swapping business keys for surrogate keys, parking late data on Unknown instead of dropping it.

**Fact table rules (ออกแบบ Fact Table):**
1. Declare the grain in one sentence. Never mix two grains; compare facts by drilling across; allocate down instead of copying.
2. Snapshot anything that's policy (price, cost, multiplier) onto the fact.
3. A list in a column becomes its own fact table.
4. Know your additivity: store additive parts, compute ratios, never sum a level over time.
5. No NULL keys (special-case rows), and facts point at dimensions, never at other facts.
6. Pick the type deliberately: transaction, periodic snapshot, accumulating snapshot, factless.

**สรุป:** warehouse มีไว้รวมหลายแหล่งให้เป็นรูปแบบเดียว นิยามกฎธุรกิจครั้งเดียว เก็บประวัติ จัดรูปให้ถามง่าย และไม่รบกวน POS
ของเราเป็น Kimball dimensional warehouse ไล่ business process → grain → dimension → fact ใช้ conformance matrix, `_key` surrogate key, SCD2 และ special-case row
กฎของ fact table: บอก grain ในประโยคเดียว ห้ามปน grain และเทียบด้วย drill across, snapshot ค่าที่เป็นนโยบาย, list ในคอลัมน์ต้องแยกเป็น fact,
รู้ว่า measure ไหนบวกได้, ไม่มี key ที่เป็น NULL และ fact ชี้ไปที่ dimension เท่านั้น, และเลือกประเภทของ fact อย่างตั้งใจ

## Check yourself (ทดสอบตัวเอง)

<details><summary>1. Why can't <code>commission_amount_thb</code> live on <code>FCT_ORDER_LINE</code>? / ทำไม <code>commission_amount_thb</code> อยู่ใน <code>FCT_ORDER_LINE</code> ไม่ได้?</summary>

It's an order-level number. On the line table it repeats once per line and every `SUM` multiplies it (3.1).
If it must be at line grain, allocate it pro-rata. / เป็นตัวเลขระดับออร์เดอร์ ถ้าอยู่ในตารางบรรทัดจะซ้ำทุกบรรทัด และ `SUM` จะคูณเพิ่ม ถ้าต้องอยู่ระดับบรรทัดจริง ๆ ให้กระจายแบบ pro-rata
</details>

<details><summary>2. The menu already has the price. Why store <code>unit_price_thb</code> on the fact too? / เมนูมีราคาอยู่แล้ว ทำไมต้องเก็บ <code>unit_price_thb</code> ใน fact อีก?</summary>

The fact records what was *paid* (including Grab's uplift). The SCD2 dimension records what was *listed*.
Without the snapshot, a price rise rewrites past revenue (3.2). / fact บันทึกราคาที่ *จ่ายจริง* (รวมส่วนที่ Grab บวกเพิ่ม) ส่วน dimension แบบ SCD2 บันทึกราคาที่ *ตั้งไว้* ถ้าไม่ snapshot การขึ้นราคาจะไปเขียนรายได้ในอดีตใหม่
</details>

<details><summary>3. Is "stock on hand" additive? / "สต็อกคงเหลือ" บวกได้ไหม?</summary>

Semi-additive: you can add it across ingredients (in baht) but never across days. Movements (`qty_delta`) are fully additive,
which is why stock on hand is derived from them (3.4, 3.6). / semi-additive: บวกข้ามวัตถุดิบได้ (เป็นเงินบาท) แต่บวกข้ามวันไม่ได้ ส่วน movement (`qty_delta`) บวกได้ทุกทาง สต็อกคงเหลือจึงคำนวณจาก movement
</details>

<details><summary>4. A Grab order has no customer. What goes in <code>customer_key</code>? / ออร์เดอร์ Grab ไม่มีลูกค้า แล้ว <code>customer_key</code> ใส่อะไร?</summary>

Row 0, *Not applicable*. A NULL would silently drop Grab's revenue from any join to `DIM_CUSTOMER` (3.5).
/ แถว 0 *ไม่เกี่ยวข้อง* ถ้าใช้ NULL รายได้ของ Grab จะหายไปจากทุก join กับ `DIM_CUSTOMER` โดยไม่มี error
</details>

<details><summary>5. Why does <code>FCT_ORDER_LINE</code> carry its own <code>date_key</code> instead of joining to <code>FCT_ORDER</code>? / ทำไม <code>FCT_ORDER_LINE</code> มี <code>date_key</code> เอง แทนที่จะ join ไปที่ <code>FCT_ORDER</code>?</summary>

A fact holds a key to every dimension it is analysed by. Reaching a dimension through another fact forces a fact-to-fact
join, which double-counts; facts are compared by drilling across instead (3.1). / fact ต้องมี key ไปหาทุก dimension ที่ใช้วิเคราะห์ ถ้าไปหา dimension ผ่าน fact อีกตัว จะต้อง join fact กับ fact ซึ่งนับซ้ำ การเทียบ fact ต้องใช้ drill across
</details>

<details><summary>6. Why surrogate keys instead of <code>PEP-L</code>? / ทำไมใช้ surrogate key แทน <code>PEP-L</code>?</summary>

SCD2 needs several rows per item, source IDs collide across systems, and the warehouse must survive a source renaming things (2.3).
/ SCD2 ต้องมีหลายแถวต่อเมนู ID จากต่างระบบชนกันได้ และ warehouse ต้องไม่พังเวลาระบบต้นทางเปลี่ยนชื่อ
</details>

In [21]:
con.close()